# 9. 轮廓提取与轮廓分析

**目标**：能从二值/边缘图中提取轮廓，并计算轮廓的几何参数与形状特征。

## 知识点清单

- **轮廓的概念**：什么是轮廓、轮廓的英文、轮廓与边缘/连通域的区别、轮廓为什么强调闭合、轮廓的来源不限于二值图——**本节新增 9.0**
- **轮廓的查找与绘制**：`findContours`、`drawContours`
- **轮廓检索模式**：`RETR_EXTERNAL`、`RETR_LIST`、`RETR_CCOMP`、`RETR_TREE`
- **轮廓近似方法**：`CHAIN_APPROX_NONE`、`CHAIN_APPROX_SIMPLE`
- **轮廓的基本参数**：面积、周长、质心、等效直径、极端点
- **轮廓的形状特征**：垂直矩形边界框、最小矩形边界框、最小外接圆、最小外接三角形、近似多边形、拟合椭圆、拟合直线、凸壳
- **轮廓平滑与去锯齿**：轮廓点集平滑（移动平均 / Savitzky-Golay）、重新采样与简化——**唯一归属本模块**
- **轮廓的属性**：宽高比、面积比、坚实度、方向、掩模、最大值/最小值及位置、灰度均值/颜色均值、内部/外部检测
- **矩不变量与形状相似性**：Hu 矩、`matchShapes`
- **附录 A**：轮廓检测算法流程、8-邻接、闭合的像素级含义

**独立性**：依赖模块 8，是形状分析的前置。

> **边界声明**：边缘检测见模块 8；形状描述子、连通域、距离变换、分水岭、骨架化见模块 10。

## 9.0 什么是轮廓

在学 `findContours` 之前，必须先回答四个最基础的问题：**什么叫轮廓？轮廓的英文是什么？轮廓为什么强调闭合？轮廓一定来自二值图吗？**

### 9.0.1 轮廓的英文

**轮廓的英文是 contour**（读音 /ˈkɒntʊə(r)/）。OpenCV 里对应的函数就是 `cv2.findContours`（find contours，查找轮廓），返回的结果通常命名为 `contours`。

### 9.0.2 轮廓的定义

> **轮廓是图像中「目标区域」与「其他区域」交界处的一圈有序像素点，首尾相连构成闭合曲线。**

**在 OpenCV 里，轮廓处理的对象是二值图。**

- `cv2.findContours` 的输入必须是**单通道二值图**（或 0/1 掩膜）；
- 二值图里只有两类像素：**前景（255）** 和 **背景（0）**；
- 轮廓就是这两类像素之间的**边界**。

这是 OpenCV 轮廓处理的第一前提——**不是灰度图，不是彩色图，是二值图**。

**为什么轮廓总是闭合的？**

因为二值图的前景是「区域」——区域天然有边界，边界天然闭合。即使前景是一条**不闭合的细线**，OpenCV 也会沿着这条线的两侧走一圈再回来，把它包成一个**闭合的细长区域**，返回的轮廓仍然是闭合的。

所以：

> **轮廓描述的是「区域的边界」，不是「线段本身」；它总是闭合的，哪怕前景看起来不闭合。**

这也解释了为什么 `contourArea`、`arcLength(closed=True)` 这些接口默认按闭合处理——**闭合是轮廓的定义性特征，不是可选项**。

（关于「闭合」在像素网格上的精确含义——为什么斜线呈锯齿、为什么需要 8-邻接——**参见附录 A.3**。）

更具体地说，在 OpenCV 的语境下：

> **轮廓是图像中目标区域与其他区域交界处的有序闭合点集，它描述的是一个闭合区域的边界。**

拆开看四个关键词：

| 关键词 | 含义 |
|--------|------|
| **目标区域 vs 其他区域** | 轮廓是把图像分成「目标」和「其他」两类后，两类之间的**边界**。注意：这里没有规定「目标区域」必须来自二值化 |
| **交界处** | 轮廓是目标区和其他区的**边界**，不是目标区内部，也不是其他区内部 |
| **有序点集** | 轮廓不是一堆散点，而是**按边界顺序排列**的点，相邻点沿着边界走 |
| **闭合** | **首尾相接，描述一个闭合区域——这是轮廓区别于边缘的核心特征** |

**关键**：定义里写的是「图像中」，不是「二值图像中」。这是有意的——**轮廓的来源不限于二值图**，详见 9.0.5。

### 9.0.3 闭合是轮廓区别于边缘的核心特征

9.0.2 已经说明：轮廓总是闭合的。这一节回答另一个问题：**「闭合」在轮廓这个概念里，到底有多重要？**

**它是轮廓区别于边缘的核心特征。**

| 概念 | 是否闭合 | 说明 |
|------|---------|------|
| **边缘（edge）** | 不一定 | 梯度突变点，可能断裂；边缘检测见模块 8 |
| **轮廓（contour）** | 是 | 区域边界，首尾相接 |
| **连通域（connected component）** | 不是线 | 是区域本身，没有「闭合」一说；连通域见模块 10 |

一句话：**边缘是「线」，连通域是「面」，轮廓是「面的闭合边界线」。**

OpenCV 多处接口都体现了「轮廓默认闭合」：

| 接口 | 体现 |
|------|------|
| `drawContours` | 默认把最后一点和第一点连起来 |
| `arcLength(cnt, closed=True)` | 参数 `closed` 就是问「要不要把首尾算一段」 |
| `approxPolyDP(cnt, eps, closed=True)` | 同样有 `closed` 参数 |
| `contourArea` | 只有闭合轮廓才有「围成的面积」 |

（关于「闭合」在像素层面的精确定义——它不是几何事实，而是「按 8-邻接规则走回起点」——**参见附录 A.3**。）

**例外情况**（唯一可能不闭合的场景）：

| 情况 | 说明 |
|------|------|
| **前景贴着图像边界** | `findContours` 会沿图像边界「切断」，返回的轮廓可能不闭合（OpenCV 会沿图像边界补一段） |
| **`arcLength(cnt, closed=False)`** | 人为选择不把首尾算一段，用于测量开放曲线的长度；但轮廓点集本身仍是 `findContours` 给的闭合边界 |
| **`CHAIN_APPROX_SIMPLE` 压缩后** | 点数变少，但闭合性不变 |

### 9.0.4 轮廓在 OpenCV 里的表示

`findContours` 返回的 `contours` 是一个 Python 列表：

```python
contours = [cnt1, cnt2, cnt3, ...]
```

**`cnt1`、`cnt2`、`cnt3` 是同一张图里发现的多个轮廓**，不是同一条轮廓的不同表示。

**每个 `cnt` 的数据结构**：

- 形状为 `(N, 1, 2)` 的 NumPy 数组；
- `N` 是这个轮廓上的点数；
- 每个点是 `[x, y]` 坐标（注意：**不是 `(x, y)` 元组，而是 `[[x, y]]` 的嵌套结构**）；
- 轮廓点按边界顺序排列，最后一个点与第一个点相邻（闭合）。

这也是为什么后面代码里取点要写 `cnt[i][0]`，取所有 x 要写 `cnt[:, :, 0]`——多出来的那一维就是为了兼容 OpenCV 的 C++ 接口。

**为什么会有多个轮廓**：

一张二值图里通常有**多个「目标区域」**。每个目标区域都有自己的边界，`findContours` 会把每个边界都找出来，放进一个列表。

比如：

| 二值图内容 | `findContours` 返回的 `contours` | 轮廓数 |
|-----------|-------------------------------|--------|
| 一个实心圆 | `[cnt1]` | 1 |
| 三个分开的方块 | `[cnt1, cnt2, cnt3]` | 3 |
| 一行文字「AB」，A、B 各有孔洞 | `[cnt1(A外), cnt2(A内), cnt3(B外), cnt4(B内)]` | 4 |
| 一个细胞，内有细胞核 | `[cnt1(细胞外), cnt2(细胞核)]` | 2 |
| 一个螺母（外圈 + 中心孔） | `[cnt1(外), cnt2(孔)]` | 2 |
| 一个方框里有洞，洞里还有小方块 | `[cnt1(外), cnt2(洞), cnt3(小方块)]` | 3 |

注意：**轮廓不一定是圆、方块这样的规则几何形状**——可以是任何封闭的形状。

**轮廓的个数 = 图里「目标区域」的个数（含孔洞、嵌套），与形状是否规则无关。**

**轮廓的个数由什么决定**：

由**检索模式**和**图像内容**共同决定：

| 因素 | 影响 |
|------|------|
| **图像内容** | 图里有几个「目标区域」（含孔洞、嵌套），就有几条边界 |
| **检索模式** | `RETR_EXTERNAL` 只取最外层；`RETR_LIST` / `RETR_CCOMP` / `RETR_TREE` 取所有 |

这正是 9.1.1 讲的四种检索模式的差别——**同一个图，换模式，`contours` 的长度就变了**。

**`contours` 是列表，所以可以遍历**：

```python
contours, hierarchy = cv2.findContours(img, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)

for i, cnt in enumerate(contours):
    print(f'第 {i+1} 个轮廓：{len(cnt)} 个点，面积 {cv2.contourArea(cnt):.1f}')
```

`contours` 就是一个普通的 Python 列表，`contours[0]` 是第一个轮廓，`contours[1]` 是第二个，`len(contours)` 是轮廓总数。

**和 `hierarchy` 的关系**：

`findContours` 还返回第二个值 `hierarchy`，它描述的是**这些轮廓之间的层级关系**：

| 返回值 | 含义 |
|--------|------|
| `contours` | 轮廓列表 `[cnt1, cnt2, ...]` |
| `hierarchy` | 层级数组，形状 `(1, N, 4)`，每个轮廓对应 `[Next, Previous, First_Child, Parent]` |

`hierarchy` 里的索引，就是 `contours` 列表的下标。所以 `hierarchy[0][i]` 描述的是 `contours[i]` 的父子兄弟关系。

**注意**：`hierarchy` 只在 `RETR_CCOMP` 和 `RETR_TREE` 下才有完整意义；`RETR_EXTERNAL` 和 `RETR_LIST` 虽然也返回它，但层级信息是退化的（全 -1 或只成一条链）。

**一句话总结**：

> **`cnt1`、`cnt2`、`cnt3` 是同一张图里发现的多个轮廓——图里有几个目标区域（含孔洞、嵌套），`contours` 里就有几条边界；具体取哪些，由检索模式决定。**

### 9.0.5 轮廓一定来自二值图吗？

**不一定。** 这个问题要分**概念层面**和**实现层面**来回答。

**结论先行**：

> **轮廓不一定来自二值图，但 `cv2.findContours` 的输入一定是二值图。**

| 说法 | 是否成立 |
|------|---------|
| 「轮廓一定是二值图中的」 | ❌ 不成立 |
| 「`cv2.findContours` 的输入一定是二值图」 | ✅ 成立 |

前者是**概念层面**的轮廓，后者是**OpenCV 这个函数**的实现约束。两者不能混为一谈。

**第一层：概念层面的轮廓，可以来自多种图**

「轮廓」的本质是**区域边界的有序点集**。只要能把图像分成「目标区域/其他区域」两类，就能提取轮廓——而分两类的方式不止二值化一种：

| 来源 | 怎么得到「目标/其他」 | 轮廓是否闭合 |
|------|---------------------|-------------|
| **二值图** | 阈值化（`threshold`、`adaptiveThreshold`） | 闭合 |
| **边缘图** | 边缘检测（Canny、Sobel） | 边缘本身不闭合，但 OpenCV 仍会把它包成闭合边界 |
| **掩膜（mask）** | 人工绘制、分割模型输出、颜色分割 | 闭合 |
| **多类别标签图** | 每个类别 vs 其他类别，做 one-vs-rest | 闭合 |
| **深度图** | 按深度阈值分层 | 闭合 |
| **灰度图直接分层** | 多个阈值区间，每层当一类 | 闭合 |

也就是说：

> **你可以从灰度图、彩色图、深度图里「定义」出轮廓，但要用 `cv2.findContours` 提取，必须先把它转成二值图/掩膜。**

**第二层：为什么 `cv2.findContours` 一定要二值图**

因为它的算法是**基于「前景区域」的边界跟踪**：

1. 扫描图像，找到第一个前景像素；
2. 从这个像素出发，沿边界走一圈，记录路径；
3. 走回起点，得到一个闭合轮廓；
4. 继续扫描，找下一个前景区域。

（完整流程——包括「标记图」「扫描穷尽 / 追踪不穷尽」——**参见附录 A.1**。）

这个算法**需要明确的「前景/背景」二值判断**，才能决定「这个像素算不算边界」「下一步往哪走」。灰度图里每个像素值都不同，算法无法判断边界在哪。

所以 `findContours` 的文档明确写：**input 是 8-bit single-channel binary image**。

**这个限制带来一个重要的实践推论**：

> **`cv2.findContours` 只规定「输入必须是二值图」，但不规定「怎么得到这张二值图」——「怎么二值」由用户决定，因为只有用户知道自己关注什么。**

换句话说：

> **转化成怎样的二值图，是用户决定的；这一步体现了用户关注什么。**

同一个 `findContours`，喂不同的二值图，出来的轮廓完全不同——**因为轮廓提取的「原料」是二值图，而二值图是用户「关注什么」的产物**。

| 用户关注什么 | 用户怎么二值化 | 得到的轮廓 |
|-------------|--------------|-----------|
| **关注细胞整体** | 阈值分割 + 形态学闭运算 | 细胞外边界 |
| **关注细胞核** | 阈值分割（高阈值）+ 面积过滤 | 细胞核边界 |
| **关注细胞膜上的蛋白** | 颜色分割（荧光通道） | 蛋白分布区域的边界 |
| **关注细胞轮廓的形状** | Canny 边缘检测 | 边缘线两侧的闭合边界 |

所以，做轮廓任务时，**真正的设计工作在「怎么生成二值图」这一步**，而不是在 `findContours` 本身。

**第三层：那「边缘图」为什么也能给 `findContours`**

模块 9 开头写的是「能从二值/边缘图中提取轮廓」。这里要澄清：

- **边缘图**（Canny 输出）本身是**二值图**——边缘为 255，非边缘为 0；
- 所以它**满足 `findContours` 的输入要求**；
- 但边缘图里的「前景」是**边缘线**，不是**实心区域**；
- `findContours` 会把这些边缘线**包成闭合的细长区域**，返回的轮廓是「线的两侧边界」，而不是「线的中心线」。

**所以「边缘图」能进 `findContours`，不是因为它不是二值图，而是因为它本身就是二值图。**

**总结**：

| 层面 | 结论 |
|------|------|
| **概念层面** | 轮廓只需「目标区域 vs 其他区域」的划分，来源不限于二值图 |
| **实现层面** | `cv2.findContours` 的输入必须是单通道二值图（或 0/1 掩膜） |
| **用户层面** | 转化成怎样的二值图由用户决定，这一步体现了用户关注什么 |
| **边缘图** | 能直接送进 `findContours`，因为它本身就是二值图 |

### 9.0.6 轮廓能做什么

补这一节还有一个好处：让读者一开始就知道「学轮廓是为了什么」。

| 用途 | 说明 |
|------|------|
| **形状分析** | 面积、周长、圆度、坚实度 |
| **形状识别** | 数顶点判三角形/四边形 |
| **目标定位** | 质心、外接框 |
| **方向估计** | `minAreaRect` 的角度 |
| **缺陷检测** | 凸包凹陷、坚实度 |
| **形状匹配** | Hu 矩、`matchShapes` |

这些正好就是模块 9 后面 9.2～9.6 的内容——**先讲「轮廓是什么」，再讲「怎么找」，最后讲「找到后能算什么」，逻辑就顺了**。

## 9.1 轮廓的查找与绘制

9.0 已经说清楚：轮廓是图像中目标区域与其他区域交界处的有序**闭合**点集；而 `cv2.findContours` 的实现要求输入是**单通道二值图**（或 0/1 掩膜）。这一节解决「怎么把它找出来、怎么画出来」。

在 OpenCV 中，查找图像轮廓通常使用 `cv2.findContours()` 函数，绘制轮廓使用 `cv2.drawContours()` 函数。理解这两个函数的关键在于掌握 `cv2.findContours()` 的两个核心参数：**轮廓检索模式（Retrieval Modes）** 与 **轮廓近似方法（Approximation Methods）**。

### 9.1.1 轮廓检索模式

检索模式决定了算法在找到轮廓后，如何处理轮廓之间的“包含关系”（即层级结构）。图像中常有嵌套图形（例如圆环里套着小圆），不同模式决定了是否记录以及记录多少层这种关系。

**1. RETR_EXTERNAL（只检测最外层轮廓）**
- **含义**：只提取最外层的轮廓，完全忽略任何嵌套在内部的轮廓。
- **为什么这样设计**：当你只关心物体的外边界，不关心内部是否有空洞或纹理时使用。
- **好处**：计算速度快，不需要处理复杂的层级逻辑；结果干净。例如做硬币计数，只想要硬币的外圆，不想关心硬币头像上的花纹细节。

**2. RETR_LIST（检测所有轮廓，不建立层级关系）**
- **含义**：提取图像中所有的轮廓（无论外层还是内层），但把它们放在一个扁平的列表里，不区分父子关系。
- **为什么这样设计**：当你只需要所有的轮廓形状，而不在乎它们之间的包含关系时。
- **好处**：获取的是一个单纯的列表，遍历方便；不会漏掉任何一个轮廓线。

**3. RETR_CCOMP（检测所有轮廓，建立两级层级）**
- **含义**：把所有轮廓组织成两层。外层轮廓为第一层，内层孔洞为第二层。如果内层孔洞里还有东西，它会重新被视为外层。
- **为什么这样设计**：主要用于处理像“甜甜圈”或“字母O”这样的结构：一个物体有一个外边界，内部有一个洞。
- **好处**：便于区分“实心物体”和“孔洞”。例如工业检测时，区分零件的外边缘和零件上的孔。

**4. RETR_TREE（检测所有轮廓，建立完整层级树）**
- **含义**：提取所有轮廓，并建立完整的“家谱图”（树状结构）。爷爷包含爸爸，爸爸包含儿子，关系完全保留。
- **为什么这样设计**：当图像极其复杂，有层层嵌套的关系时（例如俄罗斯套娃，或复杂的机械结构图）。
- **好处**：关系最全。你可以通过层级索引找到任何一个轮廓的父节点、子节点。这对于分析复杂图形逻辑非常有用。
- **核心作用**：赋予程序理解图像“空间逻辑”的能力。当任务不仅仅是“找出轮廓”，而是需要理解“这个轮廓在哪个轮廓里面”、“这个轮廓被几个轮廓包围”时，RETR_TREE 是唯一的选择。
- **典型应用**：
  - 文档版面分析（区分标题属于哪篇文章）
  - 工业质检中的缺陷检测（区分表面划痕与内部空洞）
  - 寻找环形物体（如螺母、O型密封圈）
  - 医学影像细胞计数（区分细胞核与细胞质）
  - 提取最内层或特定层级的轮廓（剥洋葱算法）

### 9.1.2 轮廓近似方法

近似方法决定了在存储轮廓点时，是“像素级”地死板存储，还是“聪明”地压缩。要理解这一点，需要先了解计算机眼里的图像是什么样的。

**预备知识**：计算机不认识“直线”、“圆”或“矩形”。它眼里的图像是一个巨大的数字矩阵（网格）。在二值化图像中，背景是0（黑），物体是1（白）。轮廓，就是白色区域和黑色区域交界处的那一圈像素点。

**1. CHAIN_APPROX_NONE（存储所有轮廓点）**
- **含义**：轮廓经过的每一个像素都被记录下来，一个不漏。即使是一段绝对笔直的线，它也会把线上所有的像素坐标都存下来。
- **为什么这样设计**：当你需要极其精确的像素级路径（比如图像分割的掩膜生成、精细的3D打印路径规划）时，不能漏掉任何一个像素，否则线条会断裂。
- **缺点**：极其占用内存。如果轮廓有几万个像素，它就会存几万个坐标点；后续处理这些海量点的速度也会变慢。

**2. CHAIN_APPROX_SIMPLE（只存储端点/关键转折点）**
- **注意**：图片表格中写“只存储端点”，这个翻译不够准确，容易让人误以为是线段的两个极端。它的真实含义是“只存储关键顶点/转折点”。
- **含义**：算法会“偷懒”，它发现一条线是直的，就只记录这条线的起点和终点（即转折处），中间的点全部扔掉。

**图示讲解（以一条倾斜直线为例）：**
假设我们在屏幕上画了一条从A点到B点的倾斜直线，它占据的像素网格如下（1代表轮廓像素）：

    像素网格示意 (放大了看)：
    (0,0) (1,0) (2,0) (3,0) (4,0)
      .     .     .     .     .
    (0,1) (1,1) (2,1) (3,1) (4,1)
      .     [A]   .     .     .    <-- 起点，记录！
    (0,2) (1,2) (2,2) (3,2) (4,2)
      .     .     .     .     .    <-- 中间点，算法发现它在直线上，丢弃！
    (0,3) (1,3) (2,3) (3,3) (4,3)
      .     .     .     .     [B]  <-- 终点，记录！

CHAIN_APPROX_SIMPLE 的做法：它只存储 (1,1) 和 (4,3) 这两个点。中间的点全部被压缩掉了。再看一个矩形的例子：如果是一个完美的矩形轮廓（由几百个像素组成），SIMPLE 只会存4个角点（顶点）。算法发现：“从左上角到右上角是一条水平直线，中间的点毫无意义，我只存左上角和右上角就够了”。

- **为什么这样设计**：绝大多数计算机视觉任务（如形状识别、测量大小）只需要知道形状的“骨架”或顶点。
- **好处**：极大节省内存（数据量可能锐减90%以上）；计算更快；抗干扰（忽略掉直线上微小的像素抖动，使线条更平滑）。
- **缺点**：丢失了直线的细节纹理。

### 9.1.3 这些点都是算法感知出来的吗？

答案是：是的，全部由算法自动计算得出。计算机并没有“眼睛”，它不知道什么是“直线”。它是如何知道哪些点是“端点/转折点”，从而进行省略的呢？这背后使用的是**道格拉斯-普克算法 (Douglas-Peucker algorithm)**。

（这个算法里「哪些点算相邻」的底层规则——**参见附录 A.2**。`findContours` 沿边界追踪时用的就是这套 8-邻接规则。）

**算法的基本逻辑（针对闭合轮廓的通俗版）：**

**第一步：先「撕开」闭合轮廓。**

闭合轮廓没有「第一个点和最后一个点」——首尾是相邻点。所以算法先在轮廓上找两个相距较远的点，把轮廓撕成两条开放折线。

**第二步：对每条折线做 Douglas-Peucker。**

1. **连接首尾**：把这条折线的起点和终点连成一条直线；
2. **找最远点**：计算折线上所有其他点到这条直线的距离；
3. **判断阈值**：如果最远距离小于阈值（比如 2 像素），说明整条折线贴近这条直线，把中间点全扔掉；如果大于阈值，说明这里有明显弯曲，保留这个点作为「转折点」，以它为界把折线分成两段，重复上述步骤。

**第三步：合并。**

两条折线的简化结果拼起来，就是闭合轮廓的近似。

**总结**：这些点不是人手动标注的，是算法根据几何数学计算出来的。算法通过计算“曲率”或“拟合误差”，自动判断哪些点对描述形状是“必要”的，哪些是“冗余”的。这就是为什么 CHAIN_APPROX_SIMPLE 能智能地压缩数据，因为它本质上是在做多边形拟合，用最少的顶点去近似原来的轮廓。

### 9.1.4 总结：为什么要这样设计？

OpenCV 的设计哲学是“根据需求提供灵活性”。

- **关于检索模式**：如果不区分层级，计算机就不知道“小圆”是大圆的一部分，还是独立的一个点。这直接影响后续的逻辑判断（比如：我要统计的是细胞数量，还是细胞内的细胞核数量？）。
- **关于近似方法**：这是一种时空权衡（Trade-off）。如果你需要精细的3D打印路径，用 NONE；如果你需要快速识别一张纸牌的花色，用 SIMPLE。

**实际应用举例**：假设你要做一个车牌识别系统。
- 你可能会用 RETR_EXTERNAL 先找到车牌的轮廓（忽略车牌上的螺丝孔）。
- 然后对车牌上的字符检测时，可能会用 RETR_TREE 来分析字符的结构（比如区分 'O' 和 'D'）。
- 为了加快识别速度，你肯定会在存储字符轮廓时使用 CHAIN_APPROX_SIMPLE，因为字符识别主要靠形状，不需要存储每一毫米的像素点。

**关于 `findContours` 内部怎么找到这些轮廓——完整算法流程参见附录 A.1。**

### 示例：轮廓查找、检索模式、近似方法

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 1. 合成带孔洞的测试图
# ============================================================
img = np.zeros((256, 256), dtype=np.uint8)
cv2.rectangle(img, (30, 30), (200, 200), 255, -1)
cv2.rectangle(img, (70, 70), (160, 160), 0, -1)
cv2.rectangle(img, (100, 100), (130, 130), 255, -1)

# ============================================================
# 2. 四种检索模式对比
# ============================================================
modes = [
    ('RETR_EXTERNAL', cv2.RETR_EXTERNAL),
    ('RETR_LIST', cv2.RETR_LIST),
    ('RETR_CCOMP', cv2.RETR_CCOMP),
    ('RETR_TREE', cv2.RETR_TREE)
]

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('原二值图'); axes[0].axis('off')

for i, (name, mode) in enumerate(modes):
    cnts, hier = cv2.findContours(img, mode, cv2.CHAIN_APPROX_SIMPLE)
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    colors = [(0,255,0), (0,0,255), (255,0,0), (0,255,255)]
    for j, cnt in enumerate(cnts):
        cv2.drawContours(vis, [cnt], -1, colors[j % len(colors)], 2)
    axes[i+1].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[i+1].set_title(f'{name}\n轮廓数: {len(cnts)}')
    axes[i+1].axis('off')
plt.tight_layout(); plt.show()

# ============================================================
# 3. NONE vs SIMPLE 对比
# ============================================================
img2 = np.zeros((256, 256), dtype=np.uint8)
cv2.circle(img2, (128, 128), 80, 255, -1)

cnts_none, _ = cv2.findContours(img2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
cnts_simple, _ = cv2.findContours(img2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f'CHAIN_APPROX_NONE 点数: {len(cnts_none[0])}')
print(f'CHAIN_APPROX_SIMPLE 点数: {len(cnts_simple[0])}')

vis_none = cv2.cvtColor(img2, cv2.COLOR_GRAY2BGR)
vis_simple = cv2.cvtColor(img2, cv2.COLOR_GRAY2BGR)
for pt in cnts_none[0]:
    cv2.circle(vis_none, tuple(pt[0]), 1, (0, 0, 255), -1)
for pt in cnts_simple[0]:
    cv2.circle(vis_simple, tuple(pt[0]), 3, (0, 255, 0), -1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.title(f'NONE 点数: {len(cnts_none[0])}')
plt.imshow(cv2.cvtColor(vis_none, cv2.COLOR_BGR2RGB)); plt.axis('off')
plt.subplot(1, 2, 2); plt.title(f'SIMPLE 点数: {len(cnts_simple[0])}')
plt.imshow(cv2.cvtColor(vis_simple, cv2.COLOR_BGR2RGB)); plt.axis('off')
plt.tight_layout(); plt.show()

**图像解读**

**一、整体概览**

这次实验用两组图展示 `findContours` 的两个关键设置：

| 组别 | 内容 | 回答的问题 |
|------|------|-----------|
| **第一组（5 张图）** | 原二值图 + 4 种检索模式 | 检索模式如何决定检测到哪些轮廓 |
| **第二组（2 张图）** | NONE vs SIMPLE 点数对比 | 近似方法如何决定轮廓点数量 |

**二、第一组：四种检索模式**

**输入图像结构**：

- **外层白色大方块**（30,30）到（200,200）；
- **中间黑色方块**（70,70）到（160,160），是外层方块里的**孔洞**；
- **内层白色小方块**（100,100）到（130,130），是孔洞里的**实心区域**。

**四种模式的检测结果**：

| 模式 | 轮廓数 | 检测到的轮廓 | 颜色标记（图中） |
|------|--------|-------------|-----------------|
| `RETR_EXTERNAL` | 1 | 只检测最外层大方块 | 绿色 |
| `RETR_LIST` | 3 | 外层大方块 + 中间黑方块 + 内层小方块 | 蓝、红、绿 |
| `RETR_CCOMP` | 3 | 同上，但建立**两级层级** | 红、蓝、绿 |
| `RETR_TREE` | 3 | 同上，但建立**完整层级树** | 绿、红、蓝 |

**关键区别**：

- **`RETR_EXTERNAL`**：**只取最外层**——中间黑方块和内层小方块全部被忽略，轮廓数只有 1；
- **`RETR_LIST`**：**取所有轮廓，不建立层级**——三个轮廓平级，轮廓数 3；
- **`RETR_CCOMP`**：**取所有轮廓，两级层级**——外层和内层分两级，轮廓数 3；
- **`RETR_TREE`**：**取所有轮廓，完整树**——层级关系最完整，轮廓数 3。

**轮廓数的判断**：

- 原图有 **3 个区域**：白色大区域、黑色孔洞、白色小区域；
- `EXTERNAL` 只取外层 → **1 个轮廓**；
- 其余三种模式取所有 → **3 个轮廓**。

**三、第二组：NONE vs SIMPLE**

**输入图像**：一个实心白色圆（半径 80）。

**两种近似方法的点数**：

| 方法 | 点数 | 说明 |
|------|------|------|
| `CHAIN_APPROX_NONE` | **452** | 存储轮廓上的**每一个像素点** |
| `CHAIN_APPROX_SIMPLE` | **244** | 只存储**端点**，直线段中间点被压缩 |

**图像观察**：

- **左图（NONE）**：红色点**密密麻麻**覆盖整个圆周，几乎连成一条线；
- **右图（SIMPLE）**：绿色点**稀疏**，只在方向变化处保留点，直线段中间的点被省略。

**为什么点数差距约一半？**

圆的轮廓在像素网格上是由**水平段、垂直段、对角段**组成的阶梯：

- **NONE**：每个阶梯像素都存 → 点数多；
- **SIMPLE**：只存每段的两个端点 → 点数少。

**这就是 `CHAIN_APPROX_SIMPLE` 的压缩原理**——直线段上的中间点不携带形状信息，可以省略。

**四、关键结论**

**这个实验证明了什么？**

> **检索模式决定“检测哪些轮廓”，近似方法决定“每个轮廓存多少点”。**

| 结论 | 证据 |
|------|------|
| **EXTERNAL 只取外层** | 轮廓数 1（只检测最外层大方块） |
| **LIST/CCOMP/TREE 取所有** | 轮廓数 3（外层 + 孔洞 + 内层） |
| **CCOMP 和 TREE 建立层级** | 层级关系不同，但轮廓数相同 |
| **NONE 存储所有点** | 点数 452 |
| **SIMPLE 只存储端点** | 点数 244（约为 NONE 的一半） |
| **SIMPLE 压缩直线段** | 方向变化处保留点，直线段中间点省略 |

**五、选择建议**

| 需求 | 推荐 |
|------|------|
| **只关心最外层轮廓** | `RETR_EXTERNAL` |
| **需要所有轮廓，不关心层级** | `RETR_LIST` |
| **需要两级层级（外轮廓 + 孔洞）** | `RETR_CCOMP` |
| **需要完整层级树（含嵌套）** | `RETR_TREE` |
| **需要精确形状，不介意点数多** | `CHAIN_APPROX_NONE` |
| **需要高效存储，形状精度可接受** | `CHAIN_APPROX_SIMPLE` |

**六、一句话总结**

> **这张图完整展示了 `findContours` 的两组关键设置：**
>
> - **检索模式**：`EXTERNAL` 只取外层（1 个轮廓），`LIST` / `CCOMP` / `TREE` 取所有（3 个轮廓），区别在于**层级关系**；
> - **近似方法**：`NONE` 存储所有点（452 个），`SIMPLE` 只存储端点（244 个），**压缩掉直线段中间点**；
> - **核心规律**：检索模式决定“检测哪些轮廓”，近似方法决定“每个轮廓存多少点”——两者独立，按需组合。

## 9.2 轮廓的基本几何量

| 几何量 | 函数 | 说明 |
|------|------|------|
| 面积 | `cv2.contourArea` | 轮廓围成的面积 |
| 周长 | `cv2.arcLength` | 轮廓的周长 |
| 质心 | `cv2.moments`（用 `m10/m00`、`m01/m00` 计算） | 轮廓的质心 |
| 等效直径 | $\sqrt{4A/\pi}$ | 面积相等的圆的直径 |
| 极端点 | 计算 | 最左、最右、最上、最下点 |

上表列出了轮廓的五个基本几何量。表格是“索引”，下面逐项解释**怎么算、为什么这样定义、在什么场景下用**。

---


### 9.2.1 面积

**定义**：轮廓所围成区域的面积。

**计算原理**：OpenCV 用**格林公式（Green's theorem）**把面积转化为沿轮廓的线积分：

$$
A = \frac{1}{2} \left| \sum_{i=1}^{n} (x_i y_{i+1} - x_{i+1} y_i) \right|
$$

其中 $(x_i, y_i)$ 是轮廓上的点，$(x_{n+1}, y_{n+1}) = (x_1, y_1)$（闭合）。

**关键说明**：

| 项目 | 说明 |
|------|------|
| **适用形状** | 任意多边形，**包括非凸轮廓**（格林公式对非凸也成立） |
| **点密度影响** | `CHAIN_APPROX_NONE` 和 `CHAIN_APPROX_SIMPLE` 算出的面积**略有差异**，因为点集不同 |
| **与 `m00` 的关系** | `cv2.moments(cnt)['m00']` 也等于面积，两者应一致 |
| **单位** | 像素² |

**典型用途**：过滤噪点轮廓（面积 < 阈值 → 丢弃）、测量目标大小、计算坚实度。

---


### 9.2.2 周长

**定义**：轮廓边界上相邻点之间的距离之和。

**计算原理**：对轮廓上相邻两点 $(x_i, y_i)$ 和 $(x_{i+1}, y_{i+1})$，累加欧氏距离：

$$
P = \sum_{i=1}^{n} \sqrt{(x_{i+1} - x_i)^2 + (y_{i+1} - y_i)^2}
$$

**关键说明**：

| 参数 | 说明 |
|------|------|
| **`closed=True`** | 闭合轮廓，**最后一点与第一点之间也算一段** |
| **`closed=False`** | 开放轮廓，不算首尾连接段 |

**与面积的区别**：

| 项目 | 面积 | 周长 |
|------|------|------|
| **量纲** | 像素² | 像素 |
| **对噪声** | 相对稳定 | **对锯齿敏感**（锯齿越多，周长越大） |
| **典型用途** | 大小、坚实度 | 圆形度、复杂度 |

**圆形度**（Circularity）：

$$
C = \frac{4\pi A}{P^2}
$$

- 圆：$C = 1$；
- 正方形：$C = \pi/4 \approx 0.785$；
- 越不规则的形状，$C$ 越接近 0。

---


### 9.2.3 质心

**定义**：轮廓所围区域的**几何中心**（不是轮廓点的平均，而是面积的加权中心）。

**计算原理**：用图像矩（moments）：

$$
M_{pq} = \sum_{x} \sum_{y} x^p y^q I(x, y)
$$

对轮廓区域，$I(x,y)=1$。质心坐标：

$$
c_x = \frac{M_{10}}{M_{00}}, \quad c_y = \frac{M_{01}}{M_{00}}
$$

其中：

| 矩 | 含义 |
|----|------|
| $M_{00}$ | 零阶矩 = **面积** |
| $M_{10}$ | 一阶矩，$x$ 方向加权和 |
| $M_{01}$ | 一阶矩，$y$ 方向加权和 |

**关键说明**：

| 项目 | 说明 |
|------|------|
| **代码写法** | `M = cv2.moments(cnt)`，然后 `cx = int(M['m10']/M['m00'])` |
| **除零保护** | 面积可能为 0（退化轮廓），要判断 `M['m00'] != 0` |
| **与极端点的区别** | 质心是**面积中心**，极端点是**边界极值点**，两者不一定重合 |

**典型用途**：目标定位、跟踪、形状匹配的参考点。

---


### 9.2.4 等效直径

**定义**：**面积相等的圆的直径**。

**推导**：

设圆的面积为 $A$，半径为 $r$：

$$
A = \pi r^2 \quad \Rightarrow \quad r = \sqrt{\frac{A}{\pi}}
$$

直径 $d = 2r$：

$$
d = 2\sqrt{\frac{A}{\pi}} = \sqrt{\frac{4A}{\pi}}
$$

**关键说明**：

| 项目 | 说明 |
|------|------|
| **本质** | 把任意形状的面积，**换算成一个等面积圆的直径** |
| **不是实际尺寸** | 对非圆形目标，等效直径**不等于**实际宽度或高度 |
| **优点** | 用一个标量快速估计目标“大小”，不受形状影响 |
| **与 `boundingRect` 的区别** | 边界框给宽高（受方向影响），等效直径只反映面积 |

**示例**：

| 形状 | 面积 | 等效直径 |
|------|------|---------|
| 圆（半径 50） | $\pi \times 50^2 \approx 7854$ | $\sqrt{4 \times 7854 / \pi} \approx 100$（正好是直径） |
| 正方形（100×100） | 10000 | $\sqrt{4 \times 10000 / \pi} \approx 112.8$ |
| 星形（面积 8000） | 8000 | $\sqrt{4 \times 8000 / \pi} \approx 100.9$ |

**注意**：正方形的等效直径（112.8）**大于**它的边长（100）——因为等面积圆比正方形“胖”。

---


### 9.2.5 极端点

**定义**：轮廓上 $x$ 最小、$x$ 最大、$y$ 最小、$y$ 最大的四个点。

**计算方式**：

```python
leftmost   = tuple(cnt[cnt[:, :, 0].argmin()][0])   # x 最小
rightmost  = tuple(cnt[cnt[:, :, 0].argmax()][0])   # x 最大
topmost    = tuple(cnt[cnt[:, :, 1].argmin()][0])   # y 最小
bottommost = tuple(cnt[cnt[:, :, 1].argmax()][0])   # y 最大

### 示例

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 1. 合成测试图：圆 + 正方形 + 星形
#    三种形状分别验证：理论值可算、形状差异明显
# ============================================================
img = np.zeros((400, 600), dtype=np.uint8)

# --- 圆：圆心 (150, 200)，半径 80 ---
cx_c, cy_c, r_c = 150, 200, 80
cv2.circle(img, (cx_c, cy_c), r_c, 255, -1)

# --- 正方形：左上 (320, 120)，边长 160 ---
x_s, y_s, side = 320, 120, 160
cv2.rectangle(img, (x_s, y_s), (x_s + side, y_s + side), 255, -1)

# --- 星形：不规则多边形 ---
pts = np.array([[150, 320], [180, 370], [240, 370], [190, 410],
                [210, 470], [150, 440], [90, 470], [110, 410],
                [60, 370], [120, 370]], np.int32)
cv2.fillPoly(img, [pts], 255)

# ============================================================
# 2. 提取轮廓
# ============================================================
contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)

# 按面积从大到小排序，方便与命名对应
contours = sorted(contours, key=cv2.contourArea, reverse=True)
names = ['正方形', '圆', '星形']   # 正方形 160×160=25600 最大

# 理论值（用于对照）
theoretical = {
    '正方形': {'area': side**2, 'perimeter': 4*side,
               'centroid': (x_s + side/2, y_s + side/2)},
    '圆':     {'area': np.pi * r_c**2, 'perimeter': 2*np.pi*r_c,
               'centroid': (cx_c, cy_c)},
    '星形':   {'area': None, 'perimeter': None, 'centroid': None},
}

# ============================================================
# 3. 逐轮廓计算五个参数
# ============================================================
img_vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

print('=' * 100)
print('9.2 轮廓基本参数：理论值 vs 实测值')
print('=' * 100)
print(f'{"形状":<8s} {"面积":>10s} {"周长":>10s} {"等效直径":>10s} '
      f'{"质心":>16s} {"极端点 L/R/T/B":>30s}')
print('-' * 100)

for cnt, name in zip(contours, names):
    # --- ① 面积 ---
    area = cv2.contourArea(cnt)

    # --- ② 周长 ---
    perimeter = cv2.arcLength(cnt, True)

    # --- ③ 等效直径 ---
    equi_diameter = np.sqrt(4 * area / np.pi)

    # --- ④ 质心 ---
    M = cv2.moments(cnt)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
    else:
        cx, cy = 0, 0

    # --- ⑤ 极端点（转成 Python int，避免打印 np.int32）---
    leftmost   = tuple(int(v) for v in cnt[cnt[:, :, 0].argmin()][0])
    rightmost  = tuple(int(v) for v in cnt[cnt[:, :, 0].argmax()][0])
    topmost    = tuple(int(v) for v in cnt[cnt[:, :, 1].argmin()][0])
    bottommost = tuple(int(v) for v in cnt[cnt[:, :, 1].argmax()][0])

    # --- 打印 ---
    print(f'{name:<8s} {area:>10.1f} {perimeter:>10.1f} '
          f'{equi_diameter:>10.1f} ({cx:>3d},{cy:>3d})   '
          f'L{leftmost} R{rightmost} T{topmost} B{bottommost}')

    # --- 可视化 ---
    cv2.drawContours(img_vis, [cnt], -1, (0, 255, 0), 2)          # 轮廓
    cv2.circle(img_vis, (cx, cy), 5, (0, 0, 255), -1)             # 质心
    cv2.putText(img_vis, 'C', (cx + 8, cy - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    for pt, color, label in [(leftmost,   (255, 0, 0), 'L'),
                             (rightmost,  (255, 0, 0), 'R'),
                             (topmost,    (255, 0, 0), 'T'),
                             (bottommost, (255, 0, 0), 'B')]:
        cv2.circle(img_vis, pt, 5, color, -1)
        cv2.putText(img_vis, label, (pt[0] + 6, pt[1] - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # 等效圆（黄色虚线）
    cv2.circle(img_vis, (cx, cy), int(equi_diameter / 2),
               (0, 255, 255), 1)

print('-' * 100)
print()

# ============================================================
# 4. 理论值对照（仅圆和正方形，星形无解析理论值）
# ============================================================
print('=' * 100)
print('理论值对照（误差 = |实测 - 理论| / 理论）')
print('=' * 100)

for cnt, name in zip(contours, names):
    th = theoretical[name]
    if th['area'] is None:
        print(f'{name}：无解析理论值，跳过对照')
        continue

    area = cv2.contourArea(cnt)
    perim = cv2.arcLength(cnt, True)
    M = cv2.moments(cnt)
    cx = M['m10'] / M['m00']
    cy = M['m01'] / M['m00']
    tx, ty = th['centroid']

    err_area = abs(area - th['area']) / th['area']
    err_perim = abs(perim - th['perimeter']) / th['perimeter']
    err_cx = abs(cx - tx)
    err_cy = abs(cy - ty)

    print(f'--- {name} ---')
    print(f'  面积：实测 {area:>8.1f}，理论 {th["area"]:>8.1f}，'
          f'相对误差 {err_area:>6.2%}')
    print(f'  周长：实测 {perim:>8.1f}，理论 {th["perimeter"]:>8.1f}，'
          f'相对误差 {err_perim:>6.2%}')
    print(f'  质心：实测 ({cx:>6.1f},{cy:>6.1f})，'
          f'理论 ({tx:>6.1f},{ty:>6.1f})，'
          f'绝对误差 ({err_cx:.2f},{err_cy:.2f}) px')

print('=' * 100)

# ============================================================
# 5. 可视化
# ============================================================
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
plt.title('轮廓基本参数可视化\n'
          '绿=轮廓  红=质心(C)  蓝=极端点(L/R/T/B)  黄=等效圆')
plt.axis('off')
plt.tight_layout()
plt.show()

**图像解读**

---

**一、整体概览**

这张图用三种形状（圆、正方形、星形）演示了轮廓的五个基本参数：

| 颜色 / 标记 | 含义 |
|------------|------|
| **绿色** | 轮廓（`findContours` 提取的边界） |
| **红色 C** | 质心（`moments` 计算的几何中心） |
| **蓝色 L / R / T / B** | 极端点（最左、最右、最上、最下） |
| **黄色虚线圆** | 等效圆（直径 = $\sqrt{4A/\pi}$） |

三种形状分别覆盖了不同的验证场景：

| 形状 | 理论值 | 验证目的 |
|------|--------|---------|
| **正方形** | 可解析 | 验证直线边界的计算精度 |
| **圆** | 可解析 | 验证曲线边界的离散化误差 |
| **星形** | 不可解析 | 验证等效直径对任意形状的适用性 |

---

**二、逐形状解读**

**正方形（右上）**

| 参数 | 实测 | 理论 | 误差 |
|------|------|------|------|
| 面积 | 25600.0 | 25600.0 | **0.00%** |
| 周长 | 640.0 | 640.0 | **0.00%** |
| 质心 | (400.0, 200.0) | (400.0, 200.0) | **0.00 px** |

观察：

- 绿轮廓**紧贴**正方形边界，四个角完整；
- 红 C **精确落在正方形中心**；
- 蓝 L / R / T / B **两两重合**在四个角上（因为正方形的角同时是最左、最上等）；
- 黄等效圆**比正方形大**——等效直径 180.5 > 边长 160，说明“等面积圆”比正方形“胖”。

结论：**直线边界的形状，面积、周长、质心全部精确**，因为 `CHAIN_APPROX_SIMPLE` 只存 4 个角点，格林公式直接给出解析结果。

**圆（左上）**

| 参数 | 实测 | 理论 | 误差 |
|------|------|------|------|
| 面积 | 19854.0 | 20106.2 | **1.25%** |
| 周长 | 529.9 | 502.7 | **5.41%** |
| 质心 | (150.0, 200.0) | (150.0, 200.0) | **0.00 px** |

观察：

- 绿轮廓**呈锯齿状**包裹圆——这是像素网格逼近连续曲线的必然结果；
- 红 C **精确落在圆心**；
- 蓝 L / R / T / B **精确落在四个方向的极值点**；
- 黄等效圆**与实际圆几乎重合**——等效直径 159.0 ≈ 实际直径 160。

结论：

- **面积误差 1.25%**：离散化导致实测面积略小于理论面积；
- **周长误差 5.41%**：**周长对锯齿最敏感**——阶梯状边界比光滑圆周长更长，但因为 `CHAIN_APPROX_SIMPLE` 压缩了直线段点，实测反而略短；
- **质心精确**：对称形状的质心与几何中心一致。

> **注意**：周长误差（5.41%）明显大于面积误差（1.25%），这符合预期——**周长是 1D 量，对边界锯齿的敏感度高于面积（2D 量）**。

**星形（下方）**

| 参数 | 实测 |
|------|------|
| 面积 | 5647.0 |
| 周长 | 448.9 |
| 等效直径 | 84.8 |
| 质心 | (150, 375) |

观察：

- 绿轮廓**完整勾勒星形**，但边缘有锯齿；
- 红 C **落在星形内部偏下**——不是星形的“几何中心”，而是**面积加权中心**；
- 蓝 T 在顶部尖角，L / R 在左右尖角，B 在底部尖角；
- 黄等效圆**比星形“胖”**——星形面积小、周长长，等效圆直径只有 84.8。

结论：

- **星形无解析理论值**，但五个参数都算出来了；
- **等效直径的价值**：给任意形状一个可比较的“大小”标量；
- **质心的含义**：面积加权中心，不是形状对称中心——星形不对称，所以 C 偏下。

---

**三、关键规律**

**规律 1：直线边界精确，曲线边界有离散化误差**

| 形状 | 面积误差 | 周长误差 |
|------|---------|---------|
| 正方形 | 0.00% | 0.00% |
| 圆 | 1.25% | 5.41% |

- 正方形的边是直线，`CHAIN_APPROX_SIMPLE` 只存角点，计算精确；
- 圆的边是阶梯，离散化导致面积、周长都有误差。

**规律 2：周长对锯齿更敏感**

| 参数 | 量纲 | 对锯齿敏感度 |
|------|------|-------------|
| 面积 | 像素² | 较低 |
| 周长 | 像素 | **较高** |

圆的周长误差（5.41%）大于面积误差（1.25%），因为周长是 1D 量，直接累加每段锯齿的长度。

**规律 3：质心精确，与形状对称性无关**

- 正方形质心 (400, 200) = 几何中心；
- 圆质心 (150, 200) = 圆心；
- 星形质心 (150, 375) ≠ 形状对称中心，但**面积加权中心**计算正确。

**规律 4：等效直径反映“面积大小”，不反映“实际尺寸”**

| 形状 | 等效直径 | 实际尺寸 |
|------|---------|---------|
| 正方形 | 180.5 | 边长 160 |
| 圆 | 159.0 | 直径 160 |
| 星形 | 84.8 | 跨度约 180 |

- 正方形等效直径 **> 边长**；
- 圆等效直径 **≈ 直径**；
- 星形等效直径 **< 跨度**（因为星形面积小）。

**等效直径只回答“面积多大”，不回答“有多宽多高”。**

---


## 9.3 轮廓的形状特征

### 9.3.1 本节的作用与定位

**三层递进关系**：

```text
    9.0 什么是轮廓
    干什么：定义“轮廓是什么”
    输出：概念、英文、闭合性、来源不限于二值图、与边缘/连通域的区别
    回答：轮廓是什么？

↓   9.1 轮廓的查找与绘制
    干什么：找到“边界点”
    输出：contours —— 一堆有序的闭合点集
    回答：边界在哪里？

↓   9.2 轮廓的基本几何量
    干什么：算“边界点围成的区域有多大、多长”
    输出：面积、周长、质心、等效直径、极端点
    回答：这个轮廓有多大？

↓   9.3 轮廓的形状特征
    干什么：用“几何模型”描述这个轮廓
    输出：八个几何模型的参数
    回答：这个轮廓能用什么几何形状描述？
```

| 小节 | 输入 | 输出 | 回答的问题 | 抽象层次 |
|------|------|------|-----------|---------|
| **9.0** | — | 概念 | 轮廓是什么 | 概念级 |
| **9.1** | 二值图 | 轮廓点集 | 边界在哪里 | 像素级 |
| **9.2** | 轮廓点集 | 标量参数 | 有多大 | 数值级 |
| **9.3** | 轮廓点集 | 几何模型 | 能用什么几何形状描述 | **几何级** |

**9.3 是 9 里抽象层次最高的一层——它不再关心“点在哪”，而是关心“这些点整体看起来像什么几何形状”。**

那“像什么几何形状”具体该怎么问？无非是问它像不像矩形、像不像圆、像不像椭圆、像不像直线、像不像三角形，以及它整体是不是凸的、能不能用更少的多边形顶点近似、凸包之外还有多少凹陷。把这几个问法各自对应到一个几何模型，就得到八个函数。

**核心问题**：轮廓点集本身**不能直接用来做判断**。

一个轮廓有 200 个点，你能从这 200 个点直接看出“它是不是正方形”吗？不能——点太多、太散，人眼和程序都很难直接判断。

**OpenCV这方面给出八个函数，就是把“200 个点”压缩成“几个几何参数”**：

| 原始轮廓 | 用 `minAreaRect` 压缩后 | 用 `approxPolyDP` 压缩后 |
|---------|------------------------|------------------------|
| 200 个点 | `((cx,cy), (w,h), angle)` = 4 个数 | 4 个顶点 |
| **无法直接判断** | **一眼看出“140×140，-60°”→ 旋转正方形** | **一眼看出“4 个顶点”→ 四边形** |

**压缩之后，判断就变得简单**：

- 顶点数 = 4 → 四边形；
- 坚实度 ≈ 1 → 凸形；
- 长短轴比例 ≈ 1 → 接近圆；
- 最小外接圆直径 ≈ 最小矩形长边 → 接近圆或正方形。

**八个函数 = 八个不同视角的“几何压缩”。**

---


### 9.3.2 八个函数总览

| 几何模型 | 对应函数                       | 一句话说明                         |
| :------- | :----------------------------- | :--------------------------------- |
| 矩形     | `boundingRect` / `minAreaRect` | 用正矩形或最小面积旋转矩形包住轮廓 |
| 圆       | `minEnclosingCircle`           | 用能包住轮廓的最小圆描述它         |
| 椭圆     | `fitEllipse`                   | 用椭圆拟合轮廓点集                 |
| 直线     | `fitLine`                      | 用直线拟合轮廓点集的主方向         |
| 三角形   | `minEnclosingTriangle`         | 用能包住轮廓的最小三角形描述它     |
| 多边形   | `approxPolyDP`                 | 用更少顶点的多边形近似轮廓         |
| 凸包     | `convexHull`                   | 用最外层凸多边形包住轮廓           |
| 凸性缺陷 | `convexityDefects`             | 描述轮廓偏离凸包的凹陷部分         |

按“模型和轮廓的关系”，这八个函数分三类：

| 类别 | 函数 | 干什么 | 特点 | 模型和轮廓的关系 | 典型用途 |
|------|------|--------|------|------------------|----------|
| 包围类 | `boundingRect` / `minAreaRect` / `minEnclosingCircle` / `minEnclosingTriangle` | 把轮廓包住 | 输出“外接模型”，一定大于等于轮廓 | 模型 ⊇ 轮廓 | 裁剪、大小、方向 |
| 拟合类 | `approxPolyDP` / `fitEllipse` / `fitLine` | 用简单模型近似轮廓 | 输出“近似模型”，可能穿过轮廓 | 模型 ≈ 轮廓 | 形状识别、简化 |
| 凸性类 | `convexHull` | 把轮廓变成凸形 | 输出“凸包”，用于坚实度和缺陷检测 | 模型 ⊇ 轮廓，且是凸的 | 凸性判断、缺陷检测 |

---


### 9.3.3 逐项解释

上表列出了轮廓的八个形状特征。表格是“索引”，下面逐项解释**怎么算、为什么这样设计、在什么场景下用**。

#### 9.3.3.1 垂直矩形边界框：`cv2.boundingRect(cnt)`

**定义**：把轮廓完全包住、**边与坐标轴平行**的最小正矩形。

**返回值**：`(x, y, w, h)`

| 返回值 | 含义 |
|--------|------|
| `x, y` | 矩形左上角坐标 |
| `w, h` | 矩形宽、高 |

**关键说明**：

| 项目 | 说明 |
|------|------|
| **方向固定** | 矩形边永远水平/垂直，**不随物体旋转** |
| **不最小** | 对斜向物体，包围框会比物体大很多 |
| **计算最快** | 只需扫描轮廓点的 $x_{\min}, x_{\max}, y_{\min}, y_{\max}$ |

**典型用途**：快速裁剪 ROI、宽高比、面积比。

**示例**：一个 45° 斜放的正方形，`boundingRect` 给出的是它的外接正矩形，面积是物体面积的 2 倍。

---


#### 9.3.3.2 最小矩形边界框：`cv2.minAreaRect(cnt)`

**定义**：把轮廓完全包住、**面积最小**的矩形，**可以旋转**。

**返回值**：`((cx, cy), (w, h), angle)`

| 返回值 | 含义 |
|--------|------|
| `(cx, cy)` | 矩形中心 |
| `(w, h)` | 矩形宽、高 |
| `angle` | 旋转角度（度） |

**关键说明**：

| 项目 | 说明 |
|------|------|
| **可旋转** | 矩形随物体方向旋转，**紧贴物体** |
| **面积最小** | 在所有可旋转矩形中面积最小 |
| **需转顶点** | 用 `cv2.boxPoints(rect)` 把 `(中心, 宽高, 角度)` 转成 4 个顶点才能绘制 |

**典型用途**：斜放物体的紧致包围、物体方向估计、最小面积裁剪。

**与 `boundingRect` 的区别**：

| 对比 | `boundingRect` | `minAreaRect` |
|------|---------------|---------------|
| 方向 | 固定水平/垂直 | **可旋转** |
| 面积 | 较大 | **最小** |
| 速度 | 快 | 稍慢 |
| 信息 | `(x, y, w, h)` | `((cx, cy), (w, h), angle)` |

---


#### 9.3.3.3 最小外接圆：`cv2.minEnclosingCircle(cnt)`

**定义**：把轮廓完全包住、**半径最小**的圆。

**返回值**：`((cx, cy), radius)`

| 返回值 | 含义 |
|--------|------|
| `(cx, cy)` | 圆心 |
| `radius` | 半径 |

**关键说明**：

| 项目 | 说明 |
|------|------|
| **旋转不变** | 圆天然旋转对称，**结果不随物体旋转变化** |
| **对凸形最紧** | 对圆形物体最紧致，对细长物体很松 |
| **比矩形更“公平”** | 不受物体方向影响 |

**典型用途**：圆形物体检测（硬币、瞳孔）、旋转不变包围、目标大小估计。

**与等效直径的区别**：

| 对比 | 等效直径 | 最小外接圆直径 |
|------|---------|---------------|
| 依据 | **面积** | **最远点距离** |
| 关系 | 等效直径 ≤ 最小外接圆直径 | — |
| 圆 | 两者相等 | — |
| 细长物体 | 等效直径小 | 最小外接圆直径大 |

---


#### 9.3.3.4 最小外接三角形：`cv2.minEnclosingTriangle(cnt)`

**定义**：把轮廓完全包住、**面积最小**的三角形。

**返回值**：`(retval, triangle)`

| 返回值 | 含义 |
|--------|------|
| `retval` | 最小三角形的面积 |
| `triangle` | 三角形的 3 个顶点，形状 `(3, 1, 2)` |

**关键说明**：

| 项目 | 说明 |
|------|------|
| **较少使用** | 比矩形、圆用得少 |
| **计算较慢** | 需要旋转卡壳类算法 |
| **顶点需转 int** | 绘制前用 `.astype(np.int32)` |
| **不一定是等边三角形** | 形状由轮廓决定，一般三边不等 |
| **不具旋转不变性** | 物体旋转后结果可能变化 |

**典型用途**：三角形物体检测、特殊形状分析。

---


#### 9.3.3.5 近似多边形：`cv2.approxPolyDP(cnt, epsilon, closed)`

**定义**：用**更少的顶点**近似原轮廓，是 Douglas-Peucker 算法的实现。

**参数**：

| 参数 | 含义 |
|------|------|
| `cnt` | 输入轮廓 |
| `epsilon` | 近似精度（最大允许偏差） |
| `closed` | 轮廓是否闭合 |

**epsilon 的作用**：

| epsilon | 效果 |
|---------|------|
| 小（如 1） | 保留更多顶点，接近原轮廓 |
| 大（如 10） | 顶点少，形状更简化 |

**关键说明**：

| 项目 | 说明 |
|------|------|
| **顶点数 = 形状线索** | 4 → 四边形，3 → 三角形 |
| **epsilon 是核心参数** | 通常取周长的 1%~5% |
| **不是 `CHAIN_APPROX_SIMPLE`** | `SIMPLE` 在 `findContours` 时压缩，`approxPolyDP` 在提取后压缩 |

**典型用途**：形状识别（数顶点判形状）、轮廓简化、去锯齿。

---


#### 9.3.3.6 拟合椭圆：`cv2.fitEllipse(cnt)`

**定义**：用**椭圆**拟合轮廓点，返回最贴合轮廓的椭圆参数。

**返回值**：`((cx, cy), (MA, ma), angle)`

| 返回值 | 含义 |
|--------|------|
| `(cx, cy)` | 椭圆中心 |
| `(MA, ma)` | 短轴、长轴 |
| `angle` | 旋转角度 |

**关键说明**：

| 项目 | 说明 |
|------|------|
| **拟合 ≠ 外接** | 椭圆**穿过**轮廓点附近，不是包住轮廓 |
| **与 `minAreaRect` 区别** | 椭圆更贴合“圆胖”形状，矩形更贴合“方正”形状 |
| **对噪声敏感** | 轮廓点少或噪声大时拟合不稳 |

**典型用途**：椭圆物体检测、形状分析、方向估计。

---


#### 9.3.3.7 拟合直线：`cv2.fitLine(cnt, distType, param, reps, aeps)`

**定义**：用**直线**拟合轮廓点，返回直线的方向和一点。

**返回值**：`(vx, vy, x0, y0)`

| 返回值 | 含义 |
|--------|------|
| `(vx, vy)` | 直线方向向量（单位向量） |
| `(x0, y0)` | 直线上一点 |

**关键参数**：

| 参数 | 含义 |
|------|------|
| `distType` | 距离类型（`DIST_L2` 最小二乘，`DIST_L1` 最小绝对偏差） |
| `param` | 距离参数（通常 0） |
| `reps, aeps` | 半径、角度精度（通常 0.01） |

**关键说明**：

| 项目 | 说明 |
|------|------|
| **拟合 ≠ 检测** | `fitLine` 需要已有轮廓点；`HoughLines` 直接从边缘检测 |
| **对噪声敏感** | 轮廓点有离群点时，`DIST_L2` 会被拉偏 |
| **返回的是无限直线** | 需自己截取线段绘制 |
| **返回值是 `(4,1)` 数组** | 需要用 `.ravel()[0]` 降成标量再参与计算 |

**典型用途**：直线度测量、方向估计、条状物体分析。

---


#### 9.3.3.8 凸壳：`cv2.convexHull(cnt)`

**定义**：包含轮廓的**最小凸多边形**（凸包）。

**返回值**：凸包顶点集，形状 (N, 1, 2)

**关键说明**：

| 项目 | 说明 |
|------|------|
| **凸包 = 最紧的凸形** | 把凹陷“填平”后的外轮廓 |
| **用于坚实度** | 坚实度 = 轮廓面积 / 凸包面积 |
| **用于缺陷检测** | `convexityDefects` 找凸包与轮廓之间的凹陷 |

**坚实度（Solidity）**：

$$
\text{Solidity} = \frac{A_{\text{contour}}}{A_{\text{convexHull}}}
$$

| 形状 | 坚实度 |
|------|--------|
| 凸形（圆、正方形） | ≈ 1 |
| 凹形（星形） | < 1 |

**典型用途**：凸性判断、缺陷检测、形状分类。

---


### 9.3.4 八个特征对照总结与典型场景

| 特征 | 函数 | 输出 | 是否旋转不变 | 典型用途 |
|------|------|------|-------------|---------|
| **垂直矩形框** | `boundingRect` | `(x,y,w,h)` | ❌ | 快速裁剪、宽高比 |
| **最小矩形框** | `minAreaRect` | `((cx,cy),(w,h),angle)` | ✅ | 紧致包围、方向 |
| **最小外接圆** | `minEnclosingCircle` | `((cx,cy),r)` | ✅ | 圆形检测、大小 |
| **最小外接三角形** | `minEnclosingTriangle` | `(area, pts)` | ❌ | 三角形检测 |
| **近似多边形** | `approxPolyDP` | 顶点集 | — | 形状识别、简化 |
| **拟合椭圆** | `fitEllipse` | `((cx,cy),(MA,ma),angle)` | ✅ | 椭圆检测、方向 |
| **拟合直线** | `fitLine` | `(vx,vy,x0,y0)` | — | 直线度、方向 |
| **凸壳** | `convexHull` | 顶点集 | ✅ | 凸性、缺陷检测 |

**用在哪：三个典型场景**

**场景 1：形状识别**

```text
轮廓 → approxPolyDP → 顶点数
  顶点数 = 3 → 三角形
  顶点数 = 4 → 四边形
  顶点数 = 5 → 五边形
  顶点数 > 10 → 圆或复杂形状
```

**场景 2：方向估计**

```text
轮廓 → minAreaRect → angle
  斜放的长方形 → 得到它的旋转角
  用于：矫正倾斜、机械臂抓取、车牌矫正
```

**场景 3：缺陷检测**

```text
轮廓 → convexHull → 坚实度
  坚实度 ≈ 1 → 无缺陷
  坚实度 < 1 → 有凹陷 → 可能有缺陷
  用于：零件缺陷、果实损伤、形状异常
```

---


### 9.3.5 本节小结

> **9.3 在干一件事：把“轮廓点集”翻译成“几何模型”。**
>
> - **9.0** 定义轮廓是什么（强调闭合、来源不限于二值图）；
> - **9.1** 找到边界点（边界在哪）；
> - **9.2** 算边界围成的区域大小（有多大）；
> - **9.3** 用八个函数把边界“装进”不同的几何形状里，得到可比较、可判断、可裁剪的几何参数（像什么形状）；
> - **八个函数 = 八个不同视角的“几何压缩”**——包围类、拟合类、凸性类；
> - **应用**：形状识别（数顶点）、方向估计（`minAreaRect` 的角度）、缺陷检测（坚实度）。
>
> **没有 9.3，轮廓就只是一堆点，没法做形状判断。**

### 示例

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 1. 合成测试图：三个形状，覆盖不同特征场景
# ============================================================
img = np.zeros((400, 600), dtype=np.uint8)

# --- 圆：验证 minEnclosingCircle / fitEllipse ---
cv2.circle(img, (150, 150), 80, 255, -1)

# --- 旋转的正方形：验证 boundingRect vs minAreaRect ---
rect_pts = cv2.boxPoints(((420, 130), (140, 140), 30))
rect_pts = np.int32(rect_pts)
cv2.fillPoly(img, [rect_pts], 255)

# --- 星形：验证 convexHull / approxPolyDP ---
star_pts = np.array([[150, 320], [180, 370], [240, 370], [190, 410],
                     [210, 470], [150, 440], [90, 470], [110, 410],
                     [60, 370], [120, 370]], np.int32)
cv2.fillPoly(img, [star_pts], 255)

# ============================================================
# 2. 提取轮廓
# ============================================================
contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
contours = sorted(contours, key=cv2.contourArea, reverse=True)

# ============================================================
# 3. 为每个轮廓计算八个形状特征，并分别可视化
# ============================================================
img_vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

COLOR_RECT    = (0, 255, 0)      # 绿：boundingRect
COLOR_MINRECT = (0, 165, 255)    # 橙：minAreaRect
COLOR_CIRCLE  = (0, 255, 255)    # 黄：minEnclosingCircle
COLOR_TRI     = (255, 0, 255)    # 紫：minEnclosingTriangle
COLOR_APPROX  = (255, 255, 0)    # 青：approxPolyDP
COLOR_ELLIPSE = (0, 0, 255)      # 红：fitEllipse
COLOR_LINE    = (255, 0, 0)      # 蓝：fitLine
COLOR_HULL    = (128, 0, 128)    # 紫红：convexHull

print('=' * 100)
print('9.3 形状特征：八个函数逐一演示')
print('=' * 100)

for i, cnt in enumerate(contours):
    print(f'\n--- 轮廓 {i + 1}：面积 = {cv2.contourArea(cnt):.1f} ---')

    # ① 垂直矩形边界框
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(img_vis, (x, y), (x + w, y + h), COLOR_RECT, 2)
    print(f'① boundingRect        ：x={x}, y={y}, w={w}, h={h}')

    # ② 最小矩形边界框
    rect = cv2.minAreaRect(cnt)
    box = np.int32(cv2.boxPoints(rect))
    cv2.drawContours(img_vis, [box], 0, COLOR_MINRECT, 2)
    print(f'② minAreaRect         ：center={rect[0]}, size={rect[1]}, '
          f'angle={rect[2]:.1f}')

    # ③ 最小外接圆
    (cx_c, cy_c), r_c = cv2.minEnclosingCircle(cnt)
    cv2.circle(img_vis, (int(cx_c), int(cy_c)), int(r_c),
               COLOR_CIRCLE, 2)
    print(f'③ minEnclosingCircle  ：center=({cx_c:.1f},{cy_c:.1f}), '
          f'r={r_c:.1f}')

    # ④ 最小外接三角形
    retval, triangle = cv2.minEnclosingTriangle(cnt)
    triangle = np.int32(triangle)
    cv2.polylines(img_vis, [triangle], True, COLOR_TRI, 2)
    print(f'④ minEnclosingTriangle：area={retval:.1f}')

    # ⑤ 近似多边形
    epsilon = 0.02 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    cv2.drawContours(img_vis, [approx], 0, COLOR_APPROX, 2)
    print(f'⑤ approxPolyDP        ：epsilon={epsilon:.1f}, '
          f'顶点数={len(approx)}')

    # ⑥ 拟合椭圆（至少需要 5 个点）
    if len(cnt) >= 5:
        ellipse = cv2.fitEllipse(cnt)
        cv2.ellipse(img_vis, ellipse, COLOR_ELLIPSE, 2)
        print(f'⑥ fitEllipse          ：center=({ellipse[0][0]:.1f},'
              f'{ellipse[0][1]:.1f}), size=({ellipse[1][0]:.1f},'
              f'{ellipse[1][1]:.1f}), angle={ellipse[2]:.1f}')
    else:
        print('⑥ fitEllipse          ：点数不足，跳过')

    # ⑦ 拟合直线（cv2.fitLine 返回 (4,1) 数组，需 .ravel()[0] 降成标量）
    vx, vy, x0, y0 = cv2.fitLine(cnt, cv2.DIST_L2, 0, 0.01, 0.01)
    vx = float(vx.ravel()[0])
    vy = float(vy.ravel()[0])
    x0 = float(x0.ravel()[0])
    y0 = float(y0.ravel()[0])
    t = 500
    pt1 = (int(x0 - vx * t), int(y0 - vy * t))
    pt2 = (int(x0 + vx * t), int(y0 + vy * t))
    cv2.line(img_vis, pt1, pt2, COLOR_LINE, 2)
    print(f'⑦ fitLine             ：方向=({vx:.3f},{vy:.3f}), '
          f'过点=({x0:.1f},{y0:.1f})')

    # ⑧ 凸壳 + 坚实度
    hull = cv2.convexHull(cnt)
    cv2.drawContours(img_vis, [hull], 0, COLOR_HULL, 2)
    area_cnt = cv2.contourArea(cnt)
    area_hull = cv2.contourArea(hull)
    solidity = area_cnt / area_hull if area_hull > 0 else 0
    print(f'⑧ convexHull          ：凸包顶点数={len(hull)}, '
          f'坚实度={solidity:.3f}')

# ============================================================
# 4. 分两部分画：左=原图，右=处理结果
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# --- 左图：原图（黑底 + 三个白块）---
axes[0].imshow(img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('① 原图\n黑底 + 三个白色实心块（圆 / 旋转正方形 / 星形）',
                  fontsize=12)
axes[0].axis('off')

# --- 右图：程序加工结果（八个形状特征叠加）---
axes[1].imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
axes[1].set_title('② 程序加工结果\n'
                  '绿=boundingRect  橙=minAreaRect  黄=minEnclosingCircle  '
                  '紫=minEnclosingTriangle\n'
                  '青=approxPolyDP  红=fitEllipse  蓝=fitLine  紫红=convexHull',
                  fontsize=10)
axes[1].axis('off')

plt.suptitle('原图 vs 程序加工：八个形状特征', fontsize=14)
plt.tight_layout()
plt.show()

**图像解读：八个形状特征逐一演示**

---

**一、整体概览**

这张图用三种形状（圆、旋转正方形、星形）演示了轮廓的八个形状特征：

| 颜色 | 特征 | 函数 |
|------|------|------|
| **绿** | 垂直矩形边界框 | `boundingRect` |
| **橙** | 最小矩形边界框 | `minAreaRect` |
| **黄** | 最小外接圆 | `minEnclosingCircle` |
| **紫** | 最小外接三角形 | `minEnclosingTriangle` |
| **青** | 近似多边形 | `approxPolyDP` |
| **红** | 拟合椭圆 | `fitEllipse` |
| **蓝** | 拟合直线 | `fitLine` |
| **紫红** | 凸壳 | `convexHull` |

三种形状分别覆盖了不同的验证场景：

| 形状 | 输出中的轮廓编号 | 验证目的 |
|------|-----------------|---------|
| **圆** | 轮廓 1 | 验证圆类特征（外接圆、拟合椭圆） |
| **旋转正方形** | 轮廓 2 | 验证旋转类特征（`boundingRect` vs `minAreaRect`） |
| **星形** | 轮廓 3 | 验证凸性特征（凸壳、坚实度） |

---

**二、逐形状解读**

**圆（左上，轮廓 1）**

| 参数 | 实测值 | 理论值 | 误差 |
|------|--------|--------|------|
| 面积 | 19854.0 | $\pi \times 80^2 \approx 20106.2$ | **1.25%** |
| `boundingRect` | 161×161 | 160×160 | +1 像素（离散化） |
| `minAreaRect` | 159.2×159.2，angle=-26.6° | 160×160 | 尺寸接近，角度无意义（圆无方向） |
| `minEnclosingCircle` | r=80.0 | 80 | **精确** |
| `fitEllipse` | 159.0×159.0，angle=0.0 | 160×160 | 接近，椭圆退化为圆 |
| `convexHull` | 60 个顶点 | — | 圆的凸包就是自己 |
| **坚实度** | **0.992** | 1.0 | 接近 1（凸形） |

**关键观察**：

- **黄外接圆精确**：r=80.0 与真实半径完全一致；
- **红拟合椭圆与黄外接圆几乎重合**：圆是椭圆的特例，长短轴相等（159.0 ≈ 159.0），角度无意义；
- **橙 `minAreaRect` 的 angle=-26.6° 是无意义的**：圆没有方向，OpenCV 给出的角度只是算法内部的一个随机取向；
- **坚实度 0.992 ≈ 1**：圆是凸形，凸包几乎等于自己。

**旋转正方形（右上，轮廓 2）**

| 参数 | 实测值 | 说明 |
|------|--------|------|
| 面积 | 19541.0 | — |
| `boundingRect` | 192×192 | 轴向包围框，**比物体大** |
| `minAreaRect` | 140.6×140.6，angle=-60° | **紧贴物体** |
| `minEnclosingCircle` | r=98.8 | 外接圆 |
| `approxPolyDP` | **4 个顶点** | 正确识别为四边形 |
| `convexHull` | 20 个顶点 | 凸包 |
| **坚实度** | **0.989** | 接近 1（凸形） |

**关键对比：`boundingRect` vs `minAreaRect`**

| 特征 | 尺寸 | 面积 | 说明 |
|------|------|------|------|
| **绿 `boundingRect`** | 192×192 | **36864** | 轴向框，物体斜放导致框很大 |
| **橙 `minAreaRect`** | 140.6×140.6 | **19768** | 旋转框，紧贴物体 |
| **实际正方形** | 140×140 | 19600 | 真实面积 |

**`minAreaRect` 面积（19768）几乎等于真实面积（19600），而 `boundingRect` 面积（36864）几乎是它的 2 倍。**

**结论**：**对旋转物体，必须用 `minAreaRect`**——`boundingRect` 会因物体倾斜而严重偏大。

**其他观察**：

- **青 `approxPolyDP` 只用 4 个顶点**：正确识别为正方形；
- **橙 `minAreaRect` 的 angle=-60°**：反映了物体的旋转方向（正方形旋转 30° 后，minAreaRect 给出的角度是 -60°，因为它定义的是“从水平轴到第一条边的角度”）；
- **坚实度 0.989 ≈ 1**：正方形是凸形。

**星形（下方，轮廓 3）**

| 参数 | 实测值 | 说明 |
|------|--------|------|
| 面积 | 5647.0 | 星形面积 |
| `boundingRect` | 181×80 | 轴向框 |
| `minAreaRect` | 79×180，angle=-90° | 旋转框 |
| `minEnclosingCircle` | r=90.0 | 外接圆 |
| `approxPolyDP` | **7 个顶点** | 星形有 10 个尖角，epsilon=9.0 合并了部分尖角 |
| `fitEllipse` | 77.8×155.9，angle=90° | 椭圆拟合 |
| `convexHull` | **11 个顶点** | 凸包填平凹陷 |
| **坚实度** | **0.651** | **明显小于 1（凹形）** |

**关键观察**：

- **黄外接圆 r=90**：比星形的任何部分都大，因为星形是凹的；
- **红拟合椭圆**：形状与星形差异大，拟合效果差——**椭圆只适合“圆胖”形状，不适合尖角形状**；
- **青 `approxPolyDP` 只保留 7 个顶点**：说明 epsilon=9.0 把一些相近的尖角合并了；
- **紫红凸壳**：**把星形的凹陷全部填平**，形成凸多边形；
- **坚实度 0.651 < 1**：这是**星形最重要的特征**——坚实度量化了“凹的程度”。

**坚实度对比**：

| 形状 | 坚实度 | 含义 |
|------|--------|------|
| 圆 | 0.992 | 凸形 |
| 正方形 | 0.989 | 凸形 |
| **星形** | **0.651** | **凹形** |

**坚实度 < 1 就意味着轮廓有凹陷。**

---

**三、八个特征的横向对比**

**特征 1：包围类（boundingRect / minAreaRect / minEnclosingCircle / minEnclosingTriangle）**

| 特征 | 圆 | 旋转正方形 | 星形 |
|------|-----|-----------|------|
| `boundingRect` | 161×161 | **192×192（偏大）** | 181×80 |
| `minAreaRect` | 159.2×159.2 | **140.6×140.6（紧）** | 79×180 |
| `minEnclosingCircle` | r=80 | r=98.8 | r=90 |
| `minEnclosingTriangle` | area=33044 | area=39191 | area=11234 |

**结论**：

- **`boundingRect` 对旋转物体严重偏大**；
- **`minAreaRect` 紧贴物体**；
- **`minEnclosingCircle` 旋转不变**；
- **`minEnclosingTriangle` 通常最松**（外接三角形面积最大）。

**特征 2：拟合类（approxPolyDP / fitEllipse / fitLine）**

| 特征 | 圆 | 旋转正方形 | 星形 |
|------|-----|-----------|------|
| `approxPolyDP` 顶点数 | 8 | **4（正确）** | 7 |
| `fitEllipse` | 159×159（圆） | 165.9×165.9 | **77.8×155.9（拟合差）** |
| `fitLine` | 水平线 | 水平线 | 水平线 |

**结论**：

- **`approxPolyDP` 对规则形状顶点数少**（正方形 4 个）；
- **`fitEllipse` 对圆最贴合，对星形最差**——椭圆适合圆胖形状，不适合尖角形状；
- **`fitLine` 返回的是“最佳拟合直线”，对非线性形状意义有限**。

**特征 3：凸性类（convexHull）**

| 形状 | 坚实度 | 凸性 |
|------|--------|------|
| 圆 | 0.992 | 凸 |
| 正方形 | 0.989 | 凸 |
| **星形** | **0.651** | **凹** |

**结论**：**坚实度 = 轮廓面积 / 凸包面积**，是判断凹性的最直接指标。

---

**四、关键规律**

**规律 1：旋转物体必须用 `minAreaRect`**

| 对比 | `boundingRect` | `minAreaRect` |
|------|---------------|---------------|
| 旋转正方形 | 192×192 | 140.6×140.6 |
| 面积比 | 36864 | 19768 |
| **偏差** | **+88%** | **+0.9%** |

**`boundingRect` 对旋转物体的面积偏差可以接近 2 倍。**

**规律 2：`fitEllipse` 只适合圆胖形状**

| 形状 | 拟合椭圆 vs 实际 | 拟合效果 |
|------|-----------------|---------|
| 圆 | 159×159 vs 160×160 | **极好** |
| 星形 | 77.8×155.9 vs 实际跨度 180 | **差** |

**规律 3：坚实度是凹性的直接指标**

| 坚实度 | 凸性 |
|--------|------|
| ≈ 1 | 凸形（圆、正方形） |
| < 1 | 凹形（星形） |
| 越小 | 凹陷越深 |

**规律 4：`approxPolyDP` 的顶点数反映形状复杂度**

| 形状 | 顶点数 | 说明 |
|------|--------|------|
| 旋转正方形 | 4 | 规则多边形 |
| 圆 | 8 | 圆用 8 个顶点近似 |
| 星形 | 7 | 部分尖角被 epsilon 合并 |

**epsilon 越大，顶点数越少。**

---

**五、参数调优建议**

**`approxPolyDP` 的 epsilon**

| epsilon | 效果 |
|---------|------|
| 小（如 1） | 顶点多，接近原轮廓 |
| 中（周长 2%） | **本例取值，平衡** |
| 大（周长 10%） | 顶点少，形状简化 |

**`fitEllipse` 的前置条件**

- **点数 ≥ 5**：少于 5 个点无法拟合椭圆；
- **点数越多，拟合越稳**：圆有 200+ 点，拟合极好；星形点少且分布不均，拟合差。

**`minEnclosingTriangle` 的计算代价**

- 比 `minAreaRect` 和 `minEnclosingCircle` **慢**；
- 结果也最松（外接三角形面积通常最大）。

---

**六、一句话总结**

> **八个形状特征，本质是“用不同的几何模型去描述轮廓”：**
>
> - **包围类**：`boundingRect`（轴向框，旋转物体偏大）、`minAreaRect`（紧致框）、`minEnclosingCircle`（旋转不变）、`minEnclosingTriangle`（最松）；
> - **拟合类**：`approxPolyDP`（顶点数反映复杂度）、`fitEllipse`（适合圆胖形状）、`fitLine`（最佳拟合直线）；
> - **凸性类**：`convexHull`（填平凹陷），坚实度是凹性的直接指标。
>
> **选择依据**：物体方向是否重要（`minAreaRect` vs `boundingRect`）、形状是否规则（`fitEllipse` vs `convexHull`）、是否需要简化（`approxPolyDP` 的 epsilon）。

## 9.4 轮廓平滑与去锯齿

### 9.4.1 问题的来源

轮廓来自二值图，而二值图的边界天然是**阶梯状**的：一条理论上的斜线，在像素网格上只能表现为“右、右、下、右、下、下”这样的锯齿。这带来三个直接后果：

| 后果 | 说明 |
|------|------|
| **周长偏大** | 锯齿越多，`arcLength` 累加的距离越大 |
| **曲率不稳** | 相邻三点方向跳变，曲率估计噪声大 |
| **拟合变差** | `fitEllipse`、`fitLine` 对锯齿敏感 |

因此，轮廓平滑的目标可以统一表述为：

> **在保留形状特征的前提下，去掉像素级锯齿。**

### 9.4.2 统一视角：把轮廓当作一维点序列

OpenCV 没有提供专门的轮廓平滑函数。所有平滑与去锯齿方法，都建立在同一个视角上：

> **把轮廓点集 $\{(x_i, y_i)\}_{i=0}^{N-1}$ 看作一条以弧长为参数的闭合一维序列，对 $x$ 和 $y$ 分别做一维信号处理。**

在这个视角下，所有方法都在回答同一个问题：

> **如何用更“干净”的点序列，替代原来的锯齿点序列？**

不同方法的区别只在于“干净”的定义：

| 方法 | “干净”的定义 |
|------|--------------|
| **移动平均** | 邻域均值 |
| **Savitzky-Golay** | 邻域内低阶多项式拟合值 |
| **重新采样** | 按弧长均匀取点 |
| **简化** | 用少量顶点近似 |

### 9.4.3 四类方法

#### 9.4.3.1 移动平均

对闭合轮廓，取窗口大小 $k$（奇数），平滑后：

$$
\hat{x}_i = \frac{1}{k} \sum_{j=-\lfloor k/2 \rfloor}^{\lfloor k/2 \rfloor} x_{(i+j) \bmod N}
$$

$y$ 同理，索引环绕取模保证首尾也平滑。

- **优点**：实现最简单，速度快；
- **缺点**：**削峰填谷**——尖角被磨圆，窗口越大越明显；
- **适用**：快速去锯齿，不关心峰谷保留。

#### 9.4.3.2 Savitzky-Golay

移动平均本质是“零阶多项式拟合”，所以会把峰谷拉平。Savitzky-Golay 改用**高阶多项式**（通常 2 阶或 3 阶）在窗口内做最小二乘拟合，取中心点的拟合值。

- **优点**：比移动平均**更能保留峰谷形状**；
- **缺点**：计算量略大；需要指定**窗口长度**和**多项式阶数**；
- **适用**：需要保留峰谷特征；
- **工具**：`scipy.signal.savgol_filter`。

#### 9.4.3.3 重新采样

按**弧长**等间隔重新取点：先累加相邻点距离得到总弧长，再按固定步长在弧长轴上插值。

- **优点**：不改变形状，只改变点密度；点分布均匀，便于曲率计算；
- **缺点**：本身不直接去锯齿，但可以作为平滑的前置步骤；
- **适用**：点数过多要降采样，点数过少要升采样，或统一不同轮廓的点数。

#### 9.4.3.4 简化（Douglas-Peucker）

就是 `cv2.approxPolyDP`，9.1 已讲过原理。它**不是平滑**，而是**用更少顶点近似**——顶点之间的直线段会“切掉”锯齿，视觉上也有平滑效果，但本质是简化。

- **优点**：顶点数可精确控制，数据量最小；
- **缺点**：会丢失细节，epsilon 越大越明显；
- **适用**：只要少量顶点，用于形状识别或存储压缩。

### 9.4.4 方法选择

| 需求 | 推荐 |
|------|------|
| 快速去锯齿，不关心峰谷 | 移动平均 |
| 保留峰谷，去锯齿 | Savitzky-Golay |
| 点数太多，要降采样 | 重新采样 |
| 只要少量顶点 | `approxPolyDP` |
| 需要精确测量面积、周长 | **不做平滑，用原始轮廓** |

### 9.4.5 注意事项

1. **平滑会改变几何量**：面积、周长、曲率都会变，平滑前先备份原始轮廓；
2. **闭合轮廓要环绕处理**：移动平均和 Savitzky-Golay 都要对索引做环绕，否则首尾断裂；
3. **窗口与阶数要合理**：窗口过大或阶数过高会过拟合，反而产生新的振荡；
4. **重新采样不改变形状**，只改变点的密度；
5. **`approxPolyDP` 的 epsilon 决定简化程度**：epsilon 越大，顶点越少，形状越简化。


### 9.4.6 计算示例：四种方法的定量对比

下面用一个带锯齿的圆轮廓，对四种方法做**统一定量对比**——既看视觉效果，也看点数与周长的变化。

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.signal import savgol_filter, medfilt
from scipy.ndimage import gaussian_filter1d, uniform_filter1d

# ================= 1. 生成带锯齿的模拟一维边界信号 =================
# 横坐标：扫描顺序（比如图像从左到右的列号）
x = np.linspace(0, 100, 100)

# 理想波浪走势（低频趋势）
y_ideal = np.sin(2 * np.pi * x / 50)

# 叠加高频噪声，模拟像素级锯齿
np.random.seed(42)
noise = np.random.normal(0, 0.15, len(x))
y_original = y_ideal + noise

# ================= 2. 定义多种一维去锯齿方法 =================
methods = {
    "方法1: 高斯滤波 (Gaussian)": gaussian_filter1d(y_original, sigma=2),
    "方法2: 中值滤波 (Median)": medfilt(y_original, kernel_size=5),
    "方法3: Savitzky-Golay": savgol_filter(y_original, window_length=11, polyorder=2),
    "方法4: 移动平均 (Moving Average)": np.convolve(y_original, np.ones(5) / 5, mode='same'),
    "方法5: 均值滤波 (Uniform)": uniform_filter1d(y_original, size=5),
    "方法6: 理想平滑 (参考基准)": y_ideal,
}

# 颜色序列，保证各方法颜色区分明显
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3']

# ================= 3. 绘制综合对比图 =================
fig = go.Figure()

# 添加原始提取曲线（灰色细线，作为背景参考）
fig.add_trace(go.Scatter(
    x=x, y=y_original,
    mode='lines',
    name='原始 NONE',
    line=dict(color='lightgray', width=1.5),
    hoverinfo='y+name'
))

# 循环添加各去锯齿方法的结果（彩色粗线）
for i, (name, y_proc) in enumerate(methods.items()):
    fig.add_trace(go.Scatter(
        x=x, y=y_proc,
        mode='lines',
        name=name,
        line=dict(color=colors[i % len(colors)], width=2.5),
        hoverinfo='y+name'
    ))

# 设置整体布局
fig.update_layout(
    title=dict(text="波浪边界去锯齿 - 多种方法综合对比", font=dict(size=18), x=0.5),
    xaxis_title="扫描顺序 (像素列)",
    yaxis_title="边界位置 (像素行)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    width=1200,
    height=600
)

fig.show()

**图像解读**

**示意图的逻辑本质**

前边给出的示意图（无论是正弦波还是阶梯波），核心是为了演示“一维信号去锯齿”的算法效果。它把二维图像处理问题，简化成了一条一维曲线。

在这条一维曲线中：
- 横坐标代表扫描的顺序（比如从左到右逐列扫描）。
- 纵坐标代表每一列中边界像素所在的位置（行号）。
- 灰色线是原始提取出来的、带有像素级跳变（锯齿）的边界曲线。
- 彩色线是经过一维滤波算法（如移动平均、高斯滤波、Savitzky-Golay）处理后的平滑曲线。

这个示意图的作用是验证算法逻辑：看哪种滤波方法能在保留波浪整体走势的前提下，最大程度抹平那些由像素离散性造成的毛刺。

**在真实图像中的应用方法**

在真实的二维图像中，应用逻辑并不是直接把整张图拿去做一维移动平均，而是分为以下几步：

第一步，确定扫描方向和提取策略。根据波浪边界的走向，选择逐列扫描（从左到右）或逐行扫描（从上到下）。前提是边界在扫描方向上必须是单值的，即每一个扫描位置只对应一个边界点，不能出现垂直回环。

第二步，提取一维边界信号。对图像的每一列（或每一行），利用边缘检测算子（如Canny、Sobel）、灰度阈值分割、或者灰度重心法，找到该列中波浪边界的精确位置。把所有列找到的边界位置按扫描顺序排列，就得到了一个一维数组。这个数组就是示意图中的“原始曲线”。

第三步，在一维数组上应用滤波算法。因为此时数据已经具有了明确的顺序关系（即扫描顺序），所以可以放心使用移动平均、高斯滤波、中值滤波、Savitzky-Golay等一维去锯齿算法。这些算法会利用相邻几个扫描位置的边界值，计算出一个平滑后的新位置。

第四步，将平滑后的边界位置映射回二维图像。把滤波后得到的新一维数组，按照原来的扫描顺序，重新写回图像中对应的列（或行）。此时，图像上的波浪边界就从原本的锯齿状变成了平滑的连续曲线。

第五步，如果不想提取一维曲线，也可以直接在二维图像上操作。这时就不能用一维移动平均，而应该使用二维滤波算法，比如二维中值滤波、二维高斯滤波、二维均值滤波或双边滤波。这些算法直接利用像素的二维空间邻域关系进行计算，不依赖扫描顺序，同样能起到去除边界锯齿的效果。

总结来说，真实图像中的应用路径有两条：一条是“先提取一维边界曲线，再对曲线做一维滤波，最后映射回图像”；另一条是“直接对二维图像做二维滤波”。示意图展示的是第一条路径中的核心环节，即一维曲线上的去锯齿处理。

## 9.5 轮廓的属性

 **属性一览**

9.3 得到的是**几何模型**（外接框、凸包等），9.5 是在这些模型基础上算**可比较的属性**。

| 属性 | 计算 | 说明 |
|------|------|------|
| **宽高比** | `w / h`（来自 `boundingRect` 或 `minAreaRect`） | 判断细长/方正 |
| **面积比** | `contourArea / (w*h)` | 轮廓占外接框的比例，反映“填充度” |
| **坚实度** | `contourArea / convexHullArea` | 凸性指标，< 1 表示有凹陷 |
| **方向** | `minAreaRect` 的 `angle` | 物体朝向 |
| **掩模** | `drawContours` 填充到全黑图 | 得到轮廓区域掩模，用于抠图 |
| **内部/外部检测** | `cv2.pointPolygonTest` | 判断某点在轮廓内、外、还是边上 |

### 9.5.1 宽高比

```python
x, y, w, h = cv2.boundingRect(cnt)
aspect_ratio = w / h
```

- ≈ 1：接近正方形/圆；
- > 1：横向细长；
- < 1：纵向细长。

### 9.5.2 面积比

```python
area = cv2.contourArea(cnt)
rect_area = w * h
extent = area / rect_area
```

- 正方形：≈ 1；
- 圆：≈ π/4 ≈ 0.785；
- 星形：明显小于 1。

### 9.5.3 坚实度

9.3 已讲：`area / convexHullArea`。凸形 ≈ 1，凹形 < 1。

### 9.5.4 方向

`minAreaRect` 返回的 `angle` 就是物体相对水平轴的方向。注意 OpenCV 不同版本对 `angle` 的定义有差异（有的返回 [-90, 0)，有的返回 [0, 90)），使用前要确认版本。

### 9.5.5 掩模

```python
mask = np.zeros(img.shape[:2], dtype=np.uint8)
cv2.drawContours(mask, [cnt], -1, 255, -1)   # 填充
result = cv2.bitwise_and(img, img, mask=mask)
```

**用途**：抠图、区域统计、把轮廓从原图分离。

### 9.5.6 内部/外部检测：`cv2.pointPolygonTest`

```python
dist = cv2.pointPolygonTest(cnt, (x, y), measureDist=True)
```

| 返回值 | 含义 |
|--------|------|
| 正数 | 点在轮廓**内部**，值为到最近边的距离 |
| 0 | 点在轮廓**边上** |
| 负数 | 点在轮廓**外部**，绝对值为到最近边的距离 |

若 `measureDist=False`，只返回 `+1 / 0 / -1`。

**用途**：判断点是否在区域内、计算点到边界的距离、碰撞检测。

### 9.5.7 计算示例

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 0. 文字工具函数
# ============================================================
def put_text_with_outline(img, text, org, font_scale=0.6,
                           color=(0, 255, 255), thickness=2):
    """黑描边 + 彩色字（用于深色背景）"""
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX,
                font_scale, (0, 0, 0), thickness + 3, cv2.LINE_AA)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX,
                font_scale, color, thickness, cv2.LINE_AA)


def put_text_plain(img, text, org, font_scale=0.6,
                    color=(0, 0, 0), thickness=2):
    """纯色文字（用于白色形状上，黑字最显眼）"""
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX,
                font_scale, color, thickness, cv2.LINE_AA)


def put_text_multiline(img, lines, org, font_scale=0.5,
                        color=(0, 255, 255), line_gap=22):
    x, y = org
    for i, line in enumerate(lines):
        put_text_with_outline(img, line, (x, y + i * line_gap),
                               font_scale, color, 1)


def measure_text_width(text, font_scale=0.55, thickness=1):
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX,
                                   font_scale, thickness)
    return tw, th


# ============================================================
# 1. 合成测试图
# ============================================================
img = np.zeros((500, 700), dtype=np.uint8)

sq_pts = cv2.boxPoints(((150, 120), (140, 140), 20))
cv2.fillPoly(img, [np.int32(sq_pts)], 255)

rect_pts = cv2.boxPoints(((450, 120), (240, 80), -30))
cv2.fillPoly(img, [np.int32(rect_pts)], 255)

cv2.circle(img, (150, 340), 80, 255, -1)

star_pts = np.array([[450, 260], [480, 310], [540, 310], [490, 350],
                     [510, 410], [450, 380], [390, 410], [410, 350],
                     [360, 310], [420, 310]], np.int32)
cv2.fillPoly(img, [star_pts], 255)

# ============================================================
# 2. 提取轮廓
# ============================================================
contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
contours = sorted(contours, key=cv2.contourArea, reverse=True)
names = ['旋转正方形', '细长矩形', '圆', '星形']

# ============================================================
# 3. 计算 6 个属性
# ============================================================
results = []
for cnt, name in zip(contours, names):
    rect = cv2.minAreaRect(cnt)
    (cx_r, cy_r), (w_r, h_r), angle = rect
    long_side, short_side = max(w_r, h_r), min(w_r, h_r)
    aspect_ratio = long_side / short_side if short_side > 0 else 0

    area = cv2.contourArea(cnt)
    rect_area = w_r * h_r
    extent = area / rect_area if rect_area > 0 else 0

    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0

    a = angle
    if a < -45:
        a += 90
    elif a > 45:
        a -= 90

    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.drawContours(mask, [cnt], -1, 255, -1)
    mask_area = int(np.count_nonzero(mask))

    M = cv2.moments(cnt)
    cx = int(M['m10'] / M['m00']) if M['m00'] != 0 else 0
    cy = int(M['m01'] / M['m00']) if M['m00'] != 0 else 0
    dist = cv2.pointPolygonTest(cnt, (float(cx), float(cy)),
                                 measureDist=True)

    results.append({
        'name': name, 'cnt': cnt, 'mask': mask,
        'rect': rect, 'hull': hull, 'cx': cx, 'cy': cy,
        'aspect': aspect_ratio, 'extent': extent,
        'solidity': solidity, 'angle': a,
        'mask_area': mask_area, 'dist': dist
    })

# ============================================================
# 4. 可视化：2x2 子图
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ------------------------------------------------------------
# 子图 1
# ------------------------------------------------------------
ax = axes[0, 0]
vis1 = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR).copy()

for r in results:
    box = np.int32(cv2.boxPoints(r['rect']))
    cv2.drawContours(vis1, [box], 0, (0, 165, 255), 2)

    bx, by, bw, bh = cv2.boundingRect(box)
    lines = [f"AR={r['aspect']:.2f}",
             f"Ex={r['extent']:.2f}",
             f"θ ={r['angle']:.1f}°"]
    font_scale = 0.55
    max_tw = 0
    for ln in lines:
        tw, th = measure_text_width(ln, font_scale, 1)
        max_tw = max(max_tw, tw)
    text_w = max_tw + 6

    tx = bx + bw + 8
    ty = by + 20
    if tx + text_w > vis1.shape[1] - 5:
        tx = bx - text_w - 8
        ty = by + 20
    if ty < 15:
        ty = 15

    put_text_multiline(vis1, lines, (tx, ty),
                        font_scale=font_scale,
                        color=(0, 255, 255), line_gap=22)

ax.imshow(cv2.cvtColor(vis1, cv2.COLOR_BGR2RGB))
ax.set_title('① 宽高比(AR) / 面积比(Ex) / 方向(θ)\n'
             '橙框 = minAreaRect    标注位于各图形右上角',
             fontsize=12, pad=10)
ax.axis('off')

# ------------------------------------------------------------
# 子图 2
# ------------------------------------------------------------
ax = axes[0, 1]
vis2 = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR).copy()
for r in results:
    cv2.drawContours(vis2, [r['hull']], 0, (128, 0, 128), 2)

    bx, by, bw, bh = cv2.boundingRect(r['hull'])
    txt = f"Sol={r['solidity']:.3f}"
    tw, th = measure_text_width(txt, 0.7, 2)

    tx = bx + bw + 8
    ty = by + 22
    if tx + tw > vis2.shape[1] - 5:
        tx = bx - tw - 8

    put_text_with_outline(vis2, txt, (tx, ty),
                           font_scale=0.7, color=(0, 255, 255),
                           thickness=2)

ax.imshow(cv2.cvtColor(vis2, cv2.COLOR_BGR2RGB))
ax.set_title('② 坚实度(Sol) = 轮廓面积 / 凸包面积\n'
             '紫红 = convexHull    凸形≈1，凹形<1',
             fontsize=12, pad=10)
ax.axis('off')

# ------------------------------------------------------------
# 子图 3
# ------------------------------------------------------------
ax = axes[1, 0]
vis3 = np.zeros((img.shape[0], img.shape[1], 3), dtype=np.uint8)
colors_mask = [(0, 255, 0), (0, 165, 255), (255, 0, 0), (255, 0, 255)]
for r, color in zip(results, colors_mask):
    vis3[r['mask'] > 0] = color
    put_text_plain(vis3, f"{r['mask_area']}",
                    (r['cx'] - 30, r['cy'] + 8),
                    font_scale=0.7, color=(255, 255, 255),
                    thickness=2)

ax.imshow(cv2.cvtColor(vis3, cv2.COLOR_BGR2RGB))
ax.set_title('③ 掩模（drawContours 填充）\n'
             '每个轮廓独立填充，数字 = 掩模像素数',
             fontsize=12, pad=10)
ax.axis('off')

# ------------------------------------------------------------
# 子图 4：内部/外部检测
# ------------------------------------------------------------
ax = axes[1, 1]
vis4 = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

# --- 4.1 质心点 + 距离：数字用纯黑色（白形状上最显眼）---
for r in results:
    cx, cy = r['cx'], r['cy']
    cv2.circle(vis4, (cx, cy), 6, (0, 0, 255), -1)

    txt = f"{r['dist']:.1f}"
    tw, th = measure_text_width(txt, 0.7, 2)
    tx = cx + 12
    if tx + tw > vis4.shape[1] - 5:
        tx = cx - tw - 12
    ty = cy - 12

    # 黑字，落在白形状上，最显眼
    put_text_plain(vis4, txt, (tx, ty),
                    font_scale=0.7, color=(0, 0, 0),
                    thickness=2)

# --- 4.2 外部测试点 + 距离：在黑色背景上，用黄字黑边 ---
star_r = results[3]
test_pt = (star_r['cx'] + 180, star_r['cy'])
d_out = cv2.pointPolygonTest(star_r['cnt'],
                              (float(test_pt[0]), float(test_pt[1])),
                              measureDist=True)
cv2.circle(vis4, test_pt, 6, (255, 0, 0), -1)

txt_out = f"外部点: {d_out:.1f}"
tw, th = measure_text_width(txt_out, 0.6, 2)
tx_out = test_pt[0] - tw // 2
ty_out = test_pt[1] + 28
if ty_out > vis4.shape[0] - 10:
    ty_out = test_pt[1] - 20
if tx_out + tw > vis4.shape[1] - 5:
    tx_out = vis4.shape[1] - tw - 8
if tx_out < 5:
    tx_out = 5

put_text_with_outline(vis4, txt_out, (tx_out, ty_out),
                       font_scale=0.6, color=(0, 255, 255),
                       thickness=2)

ax.imshow(cv2.cvtColor(vis4, cv2.COLOR_BGR2RGB))
ax.set_title('④ 内部/外部检测（pointPolygonTest）\n'
             '红点 = 质心    蓝点 = 轮廓外的测试点    '
             '数字 = 点到轮廓最近边的距离（正=内，负=外）',
             fontsize=11, pad=10)
ax.axis('off')

# ------------------------------------------------------------
# 布局
# ------------------------------------------------------------
fig.subplots_adjust(top=0.93, bottom=0.04,
                    left=0.03, right=0.97,
                    hspace=0.22, wspace=0.06)

plt.show()

## 9.6 矩不变量与形状相似性

### 9.6.1 为什么需要矩不变量

9.2 的质心、9.3 的外接框都能描述形状，但它们**对平移、旋转、缩放敏感**：

- 同一个物体平移后，质心变了；
- 旋转后，`boundingRect` 变了；
- 缩放后，面积变了。

如果要做**形状匹配**（判断两个轮廓是不是同一种形状），就需要一组**对平移、旋转、缩放都不变**的特征。这就是 **Hu 矩**。

### 9.6.2 Hu 矩

**Hu 矩是什么？**

Hu 矩是 OpenCV 提供的一种**形状特征**——它把一个轮廓压缩成 **7 个不变量**，组成一个 7 维特征向量：

```python
M = cv2.moments(cnt)
hu = cv2.HuMoments(M).flatten()   # 得到 7 个数
```

这 7 个数就是 Hu 矩的**全部内容**。它们的特点是：对**平移、旋转、缩放都不变**——这正是判形状相似所需要的。

**为什么偏偏是 7 个数？**

这不是随便定的，而是 Hu 在 1962 年从**二阶和三阶归一化中心矩**里组合出来的：

| 阶数 | 归一化中心矩 | 个数 |
|------|-------------|------|
| 二阶 | η₂₀, η₀₂, η₁₁ | 3 个 |
| 三阶 | η₃₀, η₀₃, η₂₁, η₁₂ | 4 个 |

把这 3 + 4 = 7 个矩，通过加减乘除组合成 7 个**不变量**：

| 编号 | 名称 | 特点 |
|------|------|------|
| 1 | H₁ | 平移、旋转、缩放不变 |
| 2 | H₂ | 同上 |
| 3 | H₃ | 同上 |
| 4 | H₄ | 同上 |
| 5 | H₅ | 同上 |
| 6 | H₆ | 同上 |
| 7 | H₇ | **对镜像敏感**，可区分手性（左右手） |

**为什么代码里要 `.flatten()`？**

```python
M = cv2.moments(cnt)              # 返回一个字典，包含 m00, m10, mu20, nu20 等所有矩
hu = cv2.HuMoments(M)             # 返回形状 (7, 1) 的二维数组
hu = cv2.HuMoments(M).flatten()   # flatten 后变成形状 (7,) 的一维数组
```

`.flatten()` 只是把 `(7, 1)` 压成 `(7,)`，方便后面按 `hu[0]`、`hu[1]` 这样取用，**个数还是 7 个**。

**这 7 个值用来干什么？—— 判定两个形状相似**

这 7 个值的核心用途，就是**判断两个轮廓是不是同一种形状**：

> **把两个轮廓各自压成 7 个 Hu 矩，比较它们有多接近——越接近，形状越相似。**

它能承担这个任务，靠的就是它的三条不变性：

| 不变性 | 作用 |
|--------|------|
| **平移不变** | 同一形状挪个位置，7 个值不变 → 判相似时不受位置影响 |
| **旋转不变** | 同一形状转个角度，7 个值不变 → 判相似时不受朝向影响 |
| **缩放不变** | 同一形状放大缩小，7 个值不变 → 判相似时不受大小影响 |

一句话：

> **它把「位置、朝向、大小」三件与形状无关的事屏蔽掉，只留下「形状本身」的信息——这正是判形状相似所需要的。**

**它能干什么？**

| 用途 | 说明 |
|------|------|
| **形状相似性度量** | 两个轮廓的 7 个值越接近，形状越像 |
| **形状检索** | 在形状库里，按 7 个值的差异排序，找最像的 |
| **形状粗筛** | 从大量轮廓里快速排除明显不同的 |
| **形状分类的辅助特征** | 作为特征向量，输入分类器 |
| **手性区分（第 7 个）** | 单独比较第 7 个值，区分左右手 |
| **作为 `matchShapes` 的输入** | `matchShapes` 内部就是比这 7 个值 |

**它不能干什么？**

因为它是**有损压缩**（只用二阶 + 三阶矩，只留下 7 个标量），所以它做不到下面这些：

| 做不到 | 原因 |
|--------|------|
| **区分大小** | 缩放不变——大圆和小圆值一样 |
| **区分旋转** | 旋转不变——正放和旋转 45° 的正方形值一样 |
| **区分平移** | 平移不变——同一形状在不同位置值一样 |
| **区分镜像（用 `matchShapes`）** | 前 6 个镜像不变，第 7 个取绝对值后被吞掉 |
| **区分高阶细节** | 只用到二阶 + 三阶矩——六边形和「略变形的六边形」可能很像 |
| **精确判等** | 有损压缩——不同形状可能压出相同的 7 个数 |
| **对噪声鲁棒** | 轮廓点噪声会传入 Hu 矩，导致值波动 |

**所以正确的用法是什么？**

| 用法 | 说明 |
|------|------|
| **判形状相似** | 7 个值越接近，形状越像 —— 这是主用途 |
| **粗筛 / 排序** | 从大量候选中先筛出一批「像的」，再做精细判 |
| **配合其他判据一起用** | 判「同形状」时，配合轮廓点数、顶点数、周长、面积、坚实度 |

**一句话总结**

> **Hu 矩的 7 个值是对平移、旋转、缩放不变的形状特征，用来判定两个形状是否相似：**
>
> - **同一形状的平移/旋转/缩放** → 7 个值不变 → 判为相似；
> - **不同形状** → 7 个值明显不同 → 判为不相似；
> - **能力**：算相似度、排序、检索、粗筛、区分手性；
> - **局限**：不区分大小、朝向、平移、镜像、高阶细节；
> - **定位**：**形状相似性度量工具** —— 给你「有多像」，不单独给你「是不是完全一样」。
>
> **要判「同形状」，用 Hu 矩 + `matchShapes` 做相似性判据，再配合其他特征共同判断。**

### 9.6.3 matchShapes

**一句话定位**：`matchShapes` 是 OpenCV 提供的**形状相似度打分函数**——你给它两个轮廓，它返回一个数，**越小越相似，0 表示完全相同**。

**函数签名**：

```python
score = cv2.matchShapes(cnt1, cnt2, cv2.CONTOURS_MATCH_I1, 0)
```

| 参数 | 含义 |
|------|------|
| `cnt1, cnt2` | 两个轮廓（`findContours` 返回的那种 `(N,1,2)` 数组） |
| `method` | 距离定义，三种：`CONTOURS_MATCH_I1` / `I2` / `I3` |
| `0` | 保留参数，历史遗留，通常写 `0` |

**返回值**：一个 `float`，**越小越相似**，`0` 表示两个形状完全一致（理论上）。

---

**它内部在比什么？**

核心就一句话：

> **`matchShapes` 比的是两个轮廓的 Hu 矩（9.6.2 讲的那 7 个数）。**

流程大致是：

```text
cnt1 ──► moments ──► HuMoments ──► 7 个数 h1[0..6]
cnt2 ──► moments ──► HuMoments ──► 7 个数 h2[0..6]
                                    │
                                    ▼
                        按 method 定义的公式算「距离」
                                    │
                                    ▼
                                  score
```

**score 的依据是什么？——就是那 7 个 Hu 矩，没有别的**

- 不看轮廓点在哪、不看朝向、不看大小——这些都已经被 Hu 矩屏蔽掉了；
- 只看形状本身压出来的 7 个标量。



**三种 method 的区别**，就是「怎么把这 7 个数的差异加起来」：

| method | 差异的汇总方式 | 特点 |
|--------|---------------|------|
| `CONTOURS_MATCH_I1` | 对 7 个不变量的**相对差**求和 | **最常用**，最均衡 |
| `CONTOURS_MATCH_I2` | 对 7 个不变量的**绝对差**求和 | 对大数值的矩更敏感 |
| `CONTOURS_MATCH_I3` | 取 7 个**最大相对差** | 只看差得最狠的那一个 |

三种方式的具体定义如下（$m_i = \text{sign}(h_i) \cdot \log|h_i|$，即 Hu 矩取对数后的符号化值）：

$$
I_1 = \sum_{i=1}^{7} \left| \frac{1}{m_i^A} - \frac{1}{m_i^B} \right|
$$

$$
I_2 = \sum_{i=1}^{7} \left| m_i^A - m_i^B \right|
$$

$$
I_3 = \max_i \frac{\left| m_i^A - m_i^B \right|}{\left| m_i^A \right|}
$$

**实际使用中**：绝大多数场景用 `CONTOURS_MATCH_I1` 就够了，另外两个很少用。



其中 $m_i = \text{sign}(h_i) \cdot \log|h_i|$，即 **Hu 矩取对数后的符号化值**。

**实际使用中**：绝大多数场景用 `CONTOURS_MATCH_I1` 就够了，另外两个很少用。

**为什么这样设计？三个理由**

1. **用对数**：Hu 矩的 7 个数，数值范围可能从 $10^{-30}$ 到 $10^{0}$，**跨 30 个数量级**。取对数把数量级差异拉平，让每个分量都能公平参与。
2. **用相对差 / 倒数差**：Hu 矩是归一化矩的组合，**绝对大小没有物理意义**，只有**相对变化**有意义。
3. **三种 method 是三种「重视方式」**：I1 均衡、I2 偏大数、I3 取最差。

**为什么它比「直接比 7 个数」更好用？**

| 麻烦 | `matchShapes` 怎么处理 |
|------|----------------------|
| **数值范围差异大** | 内部对 Hu 矩取对数，避免大数吃小数 |
| **浮点相等不可靠** | 不算「相等」，算「距离」 |
| **三种距离定义** | 一个参数切换，不用自己写公式 |

**调用方只需要记住一件事**：**score 越小越像**。

---

**它的三个关键性质（对应 Hu 矩的性质）**

| 性质 | 含义 | 例子 |
|------|------|------|
| **平移不变** | 物体挪位置，score 不变 | 同一个零件在画面左边和右边，score 一样 |
| **旋转不变** | 物体转个角度，score 不变 | 字母 `E` 转 30°，和原来的 `E` 仍高度相似 |
| **缩放不变** | 物体放大缩小，score 不变 | 大圆和小圆，score ≈ 0 |

**这三条性质正是它有用的原因**——做形状匹配时，通常不关心物体在哪、朝哪、多大，只关心**是不是那个形状**。

---

**它的局限（同样来自 Hu 矩）**

| 局限 | 说明 |
|------|------|
| **不考虑大小** | 大圆和小圆 score ≈ 0——如果你要区分大小，得另加面积判据 |
| **对细节不敏感** | 只用到三阶矩，对高阶细节（比如星形有几个角）不敏感 |
| **对噪声敏感** | 轮廓点噪声大时，Hu 矩波动，score 不稳 |
| **对镜像半敏感** | 前 6 个不变量镜像不变，第 7 个镜像变号；`I1/I2/I3` 一般对镜像不敏感 |
| **只有形状信息** | 不考虑颜色、纹理、位置 |

---

**正确使用姿势**

**用法 1：判断两个轮廓是否同一种形状**

```python
score = cv2.matchShapes(cnt1, cnt2, cv2.CONTOURS_MATCH_I1, 0)

if score < 0.05:
    print('很可能是同一种形状')
else:
    print('形状差异较大')
```

阈值 `0.05` 不是固定值，取决于你的数据和精度要求，通常从 `0.01 ~ 0.1` 之间调。

**用法 2：形状检索——在形状库里找最像的**

```python
best_idx = -1
best_score = float('inf')

for i, template in enumerate(templates):
    score = cv2.matchShapes(query, template, cv2.CONTOURS_MATCH_I1, 0)
    if score < best_score:
        best_score = score
        best_idx = i

print(f'最相似的模板是第 {best_idx} 个，score={best_score:.4f}')
```

**用法 3：缺陷检测——和标准形状比对**

```python
score = cv2.matchShapes(cnt_defect, cnt_standard, cv2.CONTOURS_MATCH_I1, 0)

if score > 0.1:
    print('偏离标准形状，可能有缺陷')
```

**用法 4：区分手性——单独比较第 7 个 Hu 矩**

```python
hu1 = cv2.HuMoments(cv2.moments(cnt1)).flatten()
hu2 = cv2.HuMoments(cv2.moments(cnt2)).flatten()
# 比较第 7 个（索引 6）
if np.sign(hu1[6]) != np.sign(hu2[6]):
    print('可能是镜像关系')
```

---

**它和 Hu 矩的关系（一句话）**

> **Hu 矩是「特征」，`matchShapes` 是「用 Hu 矩算距离的函数」。**
>
> **你不需要自己取 Hu 矩、取对数、写距离公式——`matchShapes` 一步到位，返回一个 score。**

**一句话总结**

> **`matchShapes` 就是「形状相似度打分器」：**
>
> - **输入**：两个轮廓；
> - **内部**：比 7 个 Hu 矩，按 `method` 算距离；
> - **输出**：score，**越小越相似，0 = 完全一致**；
> - **性质**：平移、旋转、缩放不变；
> - **用法**：设一个阈值，`score < 阈值` 就判为同一种形状。
>
> **它是 Hu 矩最常用的「落地接口」，是判形状相似度的主力工具。**

### 9.6.4 典型用途

| 用途 | 说明 |
|------|------|
| **形状检索** | 在形状库中找最相似的 |
| **手写字符识别** | 比较字符轮廓 |
| **零件分类** | 判断零件是否同一种 |
| **缺陷检测** | 与标准形状比对，偏离大则有缺陷 |

> **注意**：`matchShapes` 只比较**形状**，不考虑颜色、纹理、大小。如果两个形状相似但大小差很多，`matchShapes` 仍会给出很小的值——这是 Hu 矩缩放不变性的体现，也是它的局限。

### 9.6.5 示例：Hu 矩、matchShapes 与形状相似性

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 1. 合成测试图：
#    同一个形状（正方形）的 4 种变换 + 2 个不同形状 + 1 个镜像
# ============================================================
img = np.zeros((600, 900), dtype=np.uint8)

# ------------------------------------------------------------
# 边画边记录每个形状的「真名 + 中心」
#   之后用质心匹配，避免"按面积排序猜名字"的 bug
# ------------------------------------------------------------
shape_truth = []

def add_shape(name, cx, cy):
    shape_truth.append({'name': name, 'cx': cx, 'cy': cy})

# --- 基准正方形：中心 (120, 120)，边长 100 ---
cv2.rectangle(img, (70, 70), (170, 170), 255, -1)
add_shape('基准正方形', 120, 120)

# --- 平移正方形：中心 (320, 120) ---
cv2.rectangle(img, (270, 70), (370, 170), 255, -1)
add_shape('平移正方形', 320, 120)

# --- 旋转正方形：中心 (520, 120)，边长 100，旋转 45° ---
rect_pts = cv2.boxPoints(((520, 120), (100, 100), 45))
cv2.fillPoly(img, [np.int32(rect_pts)], 255)
add_shape('旋转正方形', 520, 120)

# --- 缩放正方形：中心 (720, 120)，边长 50 ---
cv2.rectangle(img, (695, 95), (745, 145), 255, -1)
add_shape('缩放正方形', 720, 120)

# --- 圆：中心 (120, 350)，半径 60 ---
cv2.circle(img, (120, 350), 60, 255, -1)
add_shape('圆', 120, 350)

# --- 星形：中心 (400, 350) 附近 ---
star_pts = np.array([[400, 280], [420, 320], [465, 320], [430, 350],
                     [445, 395], [400, 370], [355, 395], [370, 350],
                     [335, 320], [380, 320]], np.int32)
cv2.fillPoly(img, [star_pts], 255)
add_shape('星形', 400, 350)

# --- 镜像正方形：中心 (750, 330) ---
cv2.rectangle(img, (700, 280), (800, 380), 255, -1)
add_shape('镜像正方形', 750, 330)

# ============================================================
# 2. 提取轮廓
# ============================================================
contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)

def nearest_shape_name(cnt):
    M = cv2.moments(cnt)
    if M['m00'] == 0:
        return '未知'
    cx = M['m10'] / M['m00']
    cy = M['m01'] / M['m00']
    best_name = '未知'
    best_dist = float('inf')
    for s in shape_truth:
        d = (s['cx'] - cx) ** 2 + (s['cy'] - cy) ** 2
        if d < best_dist:
            best_dist = d
            best_name = s['name']
    return best_name

named_contours = [(nearest_shape_name(cnt), cnt) for cnt in contours]

base_cnt = None
for name, cnt in named_contours:
    if name == '基准正方形':
        base_cnt = cnt
        break

# ============================================================
# 3. 计算 Hu 矩（保留，用于控制台输出）
# ============================================================
print('=' * 110)
print('9.6 Hu 矩：每个轮廓的 7 个不变量（取 log10 后的值）')
print('=' * 110)

hu_list = []
for name, cnt in named_contours:
    M = cv2.moments(cnt)
    hu = cv2.HuMoments(M).flatten()
    hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)
    hu_list.append((name, hu_log))

order = ['基准正方形', '平移正方形', '旋转正方形',
         '缩放正方形', '圆', '星形', '镜像正方形']
hu_sorted = sorted(hu_list, key=lambda x: order.index(x[0]))

print(f'{"形状":<12s}  '
      f'{"H1":>8s} {"H2":>8s} {"H3":>8s} {"H4":>8s} '
      f'{"H5":>8s} {"H6":>8s} {"H7":>8s}')
print('-' * 110)
for name, hu_log in hu_sorted:
    print(f'{name:<12s}  ' +
          ' '.join(f'{v:>8.3f}' for v in hu_log))
print('=' * 110)
print()

# ============================================================
# 4. 计算 matchShapes score（与基准正方形比较）
# ============================================================
print('=' * 110)
print('9.6 matchShapes：每个轮廓 vs 基准正方形的 score')
print('     score 越小越相似，0 表示完全一致')
print('=' * 110)
print(f'{"形状":<12s} {"I1":>12s} {"I2":>12s} {"I3":>12s}   {"判定（I1<0.05）"}')
print('-' * 110)

scores = []
for name, cnt in named_contours:
    s1 = cv2.matchShapes(base_cnt, cnt, cv2.CONTOURS_MATCH_I1, 0)
    s2 = cv2.matchShapes(base_cnt, cnt, cv2.CONTOURS_MATCH_I2, 0)
    s3 = cv2.matchShapes(base_cnt, cnt, cv2.CONTOURS_MATCH_I3, 0)
    scores.append((name, s1, s2, s3))

for name, s1, s2, s3 in sorted(scores, key=lambda x: x[1]):
    judge = '同一种形状' if s1 < 0.05 else '不同形状'
    print(f'{name:<12s} {s1:>12.6f} {s2:>12.6f} {s3:>12.6f}   {judge}')

print('=' * 110)
print()

# ============================================================
# 5. 可视化：2 个子图
#    左：原图 + 所有轮廓 + 名称（红框 = 基准正方形）
#    右：matchShapes score 条形图（方案 A）
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ---------- 子图 1：原图 + 所有轮廓 ----------
vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
for name, cnt in named_contours:
    if name == '基准正方形':
        cv2.drawContours(vis, [cnt], -1, (0, 0, 255), 3)
    else:
        cv2.drawContours(vis, [cnt], -1, (0, 255, 0), 2)

    x, y, w, h = cv2.boundingRect(cnt)
    tw, th = cv2.getTextSize(name, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]
    tx = x + (w - tw) // 2
    ty = y - 8
    if ty < 15:
        ty = y + th + 12

    cv2.putText(vis, name, (tx, ty),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(vis, name, (tx, ty),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1, cv2.LINE_AA)

axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
axes[0].set_title('① 原图：7 个测试形状\n'
                  '红框 = 基准正方形（对比模板）  绿框 = 其它形状',
                  fontsize=13)
axes[0].axis('off')

# ---------- 子图 2：matchShapes score 条形图（方案 A）----------
# 收集 score（用 I1）
score_map = {}
for name, s1, s2, s3 in scores:
    score_map[name] = s1

# 按 score 从小到大排序
order_by_score = sorted(score_map.keys(), key=lambda n: score_map[n])
x_pos = np.arange(len(order_by_score))
y_vals = [score_map[n] for n in order_by_score]

# 基准正方形用红色，其它用蓝色
colors = ['#d62728' if n == '基准正方形' else '#1f77b4'
          for n in order_by_score]

bars = axes[1].bar(x_pos, y_vals, color=colors,
                   edgecolor='black', linewidth=1.0)

# 阈值线
THRESHOLD = 0.05
axes[1].axhline(THRESHOLD, color='green', linestyle='--',
                linewidth=1.5, label=f'阈值 = {THRESHOLD}')

# 在每根柱子上写 score 值
for bar, val in zip(bars, y_vals):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.002,
                 f'{val:.4f}',
                 ha='center', va='bottom', fontsize=10)

axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(order_by_score, rotation=30, ha='right',
                        fontsize=11)
axes[1].set_ylabel('matchShapes score（越小越相似）', fontsize=12)
axes[1].set_title('② matchShapes score：各形状 vs 基准正方形\n'
                  '绿虚线 = 阈值 0.05；红柱 = 基准正方形自身（score = 0）',
                  fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 附录

### 附录 A：轮廓检测算法细节

本附录补充 `cv2.findContours` 内部的算法细节——正文里提到但没展开的「扫描 / 追踪」「8-邻接」「闭合的像素级含义」。

正文首次出现的引用位置：

| 正文位置 | 引用附录 |
|---------|---------|
| 9.0.2「为什么轮廓总是闭合的」 | 附录 A.3 |
| 9.0.3「闭合是轮廓区别于边缘的核心特征」 | 附录 A.3 |
| 9.0.5「第二层：为什么 `cv2.findContours` 一定要二值图」 | 附录 A.1 |
| 9.1.3「这些点都是算法感知出来的吗？」 | 附录 A.2 |
| 9.1.4 末尾 | 附录 A.1 |

### A.1 轮廓检测算法流程

#### A.1.1 两个阶段：扫描 + 追踪

`cv2.findContours` 内部大致分两个阶段：

| 阶段 | 做什么 | 是否穷尽 | 复杂度 |
|------|--------|---------|--------|
| **① 扫描（scan）** | 从上到下、从左到右，逐个像素找**还没被访问过**的前景像素 | **是**——每个像素都要「看」一遍 | O(W × H) |
| **② 追踪（trace）** | 从找到的前景像素出发，沿 8-邻接走一圈，标记走过的点 | **否**——只沿边界走 | O(P)，P 为边界点总数 |

**总复杂度**：O(W × H + P)——扫描主导，追踪只走边界。

#### A.1.2 完整流程

```text
1. 初始化标记图：与输入同尺寸，全 0（未访问）
2. 逐像素扫描（从上到下、从左到右）：
   for 每个像素 (x, y):
       if 是前景像素 且 标记图(x,y) == 未访问:
           从 (x,y) 开始追踪
           ┌─────────────────────────────┐
           │ 追踪循环：                   │
           │   当前点标记为「已追踪」      │
           │   在 8 个邻居里找下一个边界点 │
           │   走过去                     │
           │   直到走回起点               │
           └─────────────────────────────┘
           保存这个轮廓
3. 继续扫描，跳过已标记的点
4. 直到遍历完整张图
5. 返回所有轮廓 + 层级
```

#### A.1.3 关键机制 1：标记图

**不是「像素列表」，而是与图像同尺寸的状态图**：

| 状态 | 含义 |
|------|------|
| **未访问** | 还没被追踪过 |
| **已追踪** | 已作为某个轮廓的边界点被记录 |

**作用**：

- **避免重复**：一个前景像素最多被追踪一次；
- **避免死循环**：追踪时不会走回已经走过的点；
- **保证找全**：扫描结束后，所有前景像素要么被追踪过，要么已被标记为「属于某个轮廓」；
- **支持层级**：标记图帮助算法区分「外边界」和「孔洞边界」。

**这也是 `findContours` 要求单通道二值图的工程理由之一**——标记图与输入同尺寸，单通道最省。

#### A.1.4 关键机制 2：走回起点

追踪的终止条件是**走回起点**，不是「最后一点和起点相邻」。

- 追踪的每一步都往边界上「下一个」像素走；
- 当「下一个」是**起始点**时，追踪结束；
- 这时轮廓点集**首尾相接**，是闭合的。

**所以「走回起点」才是闭合的判据**，「相邻」只是它的一个必要条件。

#### A.1.5 常见误解

| 说法 | 是否准确 |
|------|---------|
| 「对前景中每个像素都追踪」 | ❌ 不是——每个前景像素**最多被追踪一次** |
| 「对每个还没被访问过的前景像素作为新起点追踪」 | ✅ 准确 |
| 「最后一点和起点相邻就是轮廓」 | ⚠️ 不准确——判据是「**走回起点**」 |
| 「一个前景区域可能对应多条边界追踪」 | ✅ 准确（比如细线的两侧） |

### A.2 相邻的定义：8-邻接

#### A.2.1 两种邻接定义

一个像素周围有 8 个邻居，但「相邻」有两种标准：

| 邻接类型 | 邻居 | OpenCV 用哪个 |
|---------|------|--------------|
| **4-邻接** | 上、下、左、右 | ❌ |
| **8-邻接** | 上下左右 + 四个对角 | ✅ |

用图表示（中心像素为 P）：

```text
4-邻接：            8-邻接：
  . 上 .             左上 上 右上
  左 P 右             左  P  右
  . 下 .             左下 下 右下
```

#### A.2.2 为什么用 8-邻接

**只有它能把斜向边界连成连续路径，轮廓才能闭合。**

- 如果只用 4-邻接，一条**斜线**在像素网格上会断——对角方向的像素彼此「不相邻」，路径连不起来；
- 8-邻接把对角也算相邻，斜线才能连成连续路径。

#### A.2.3 硬编码，不可配置

**`cv2.findContours` 的函数签名里没有「邻接类型」参数**——它内部**硬编码为 8-邻接**：

```python
cv2.findContours(image, mode, method[, contours[, hierarchy[, offset]]])
```

| 参数 | 含义 |
|------|------|
| `image` | 输入二值图 |
| `mode` | 轮廓检索模式（`RETR_EXTERNAL` 等） |
| `method` | 轮廓近似方法（`CHAIN_APPROX_NONE` / `SIMPLE`） |
| `offset` | 轮廓点偏移（可选） |

**没有「4-邻接 / 8-邻接」这个参数。** OpenCV 源码里，`findContours` 内部使用的邻接查找数组是固定的 8 方向版本。

**如果确实需要 4-邻接**：`findContours` 做不到，但 `connectedComponents` 有 `connectivity` 参数可以指定 4 或 8——不过它做的是连通域标记，不是轮廓提取。

#### A.2.4 方向与距离

在 8-邻接下，从一个像素到相邻像素，方向有 8 种：

| 方向 | 偏移 (dx, dy) | 距离 |
|------|--------------|------|
| 右 | (+1, 0) | 1 |
| 左 | (-1, 0) | 1 |
| 下 | (0, +1) | 1 |
| 上 | (0, -1) | 1 |
| 右下 | (+1, +1) | √2 |
| 左下 | (-1, +1) | √2 |
| 右上 | (+1, -1) | √2 |
| 左上 | (-1, -1) | √2 |

**`CHAIN_APPROX_SIMPLE` 的压缩规则**：

- 如果连续多个像素的**方向相同**（比如一直往右），只存首尾；
- 方向一变（比如从「右」变「右下」），就存一个转折点。

#### A.2.5 4-邻接和 8-邻接的「连通性悖论」

在二值图里，如果前景用 8-邻接，背景也必须用 8-邻接，会出现**拓扑矛盾**：

- 一条 8-邻接的斜线，会「穿过」另一条 8-邻接的斜线，导致前景和背景连通性不一致；
- 解决办法：**前景和背景用不同的邻接**（一个用 4-邻接，一个用 8-邻接）。

OpenCV 的 `findContours` 内部处理了这个细节，**对外表现为 8-邻接追踪**，用户不用管这个悖论。

### A.3 闭合在像素层面的含义

#### A.3.1 问题

一条理论上的**斜线**，在像素网格上根本画不出「斜」——只能画成「右、右、下、右、下、下」这样的**阶梯**。

那么问题来了：

> **这条阶梯状的点集，什么时候算「走回了起点」，也就是封闭？**

这**不是几何上显然的**——它取决于算法采用什么样的「邻接规则」和「走法规则」。

#### A.3.2 闭合是拓扑概念，不是几何概念

「闭合」在像素层面**不是几何事实**，而是**算法按 8-邻接规则判断出来的**：

- 一条斜线在像素网格上**必然呈锯齿**；
- 只有 8-邻接才能把斜线连成连续路径；
- 追踪走回起点，轮廓就「闭合」了——**但几何上仍是锯齿**。

**所以**：

> **「封闭」是拓扑概念（走回起点），不是几何概念（形状光滑）。**

#### A.3.3 这解释了几件事

| 现象 | 原因 |
|------|------|
| **圆的周长实测比理论大** | 阶梯状边界的路径长度 > 光滑圆周长度 |
| **`CHAIN_APPROX_SIMPLE` 能压缩** | 8-邻接下，对角段是「直线」，只存首尾 |
| **`approxPolyDP` 能简化** | 把拓扑上连续的锯齿，近似成几何上的直线段 |
| **9.4 要单独讲平滑** | 拓扑封闭 ≠ 几何光滑，平滑要另外做 |
| **斜向物体的 `boundingRect` 偏大** | 斜向物体在像素网格上的外接框，受锯齿影响 |

#### A.3.4 一句话总结

> **「封闭性」在像素层面不是几何事实，而是「按 8-邻接规则，沿 0/255 交界走一圈能回到起点」——这条规则是算法定义的，不是图像自带的。**
>
> **斜向边界在像素网格上必然呈锯齿；8-邻接让它在拓扑上连续（封闭），但几何上仍是阶梯。这正是 `CHAIN_APPROX_SIMPLE` 能压缩、`approxPolyDP` 能简化、9.4 要单独讲平滑的根本原因。**